# Curso de Machine Learning Miercoles 17/21 hs - 2026
# Parte 1 — Análisis y limpieza de datos

Primero de dos cuadernos. Este recorre el **pipeline de preparación de datos** completo: desde la
carga del archivo hasta dejar los conjuntos de entrenamiento y prueba listos para modelar.

**Qué hace este cuaderno**

| # | Etapa | Dónde |
|---|---|---|
| 1 | Selección del dataset: origen, problema y justificación | punto 1 |
| 2 | Análisis exploratorio (EDA): distribuciones, faltantes, correlaciones, gráficos | punto 3 |
| 3 | Limpieza: nulos, duplicados, outliers | punto 4 |
| 4 | Transformaciones: codificación de categóricas y escalado de numéricas | punto 5 |
| 5 | Selección de variables relevantes | punto 6 |
| 6 | División en entrenamiento y prueba | punto 7 |

**Qué NO hace**

No entrena ningún modelo. Eso es la Parte 2. Acá el objetivo es dejar los datos en condiciones, y
que cada decisión tomada en el camino quede justificada por escrito.

**El dataset**

Un archivo Excel con 171 filas y 8 columnas. La primera, `TAG`, es la variable objetivo y tiene
dos valores posibles: `AFI` y `NEG`. Las otras siete son numéricas y se llaman `Topico 0` a
`Topico 6`. El punto 1 lo describe en detalle.

**Cómo usarlo**

Las celdas se corren de arriba hacia abajo, en orden. Si algo se comporta raro,
`Entorno de ejecución → Reiniciar y ejecutar todo`.

---

# Índice

### Parte 0 — Preliminares
- [0.1 El entorno de trabajo: Google Colab](#sec-0-1)
- [0.2 Cargar el dataset](#sec-0-2)
  - [0.2.1 Las cuatro opciones](#sec-0-2-1)
  - [0.2.2 Leer el archivo](#sec-0-2-2)
  - [0.2.3 Si la lectura sale mal](#sec-0-2-3)
- [0.3 Configuración global](#sec-0-3)

### 1. Selección del dataset
- [1.1 Origen de los datos](#sec-1-1)
- [1.2 De texto a números: el modelo de tópicos](#sec-1-2)
- [1.3 La búsqueda de la mejor separación](#sec-1-3)
- [1.4 El problema que aborda](#sec-1-4)
- [1.5 Justificación de la elección](#sec-1-5)
- [1.6 Ficha técnica](#sec-1-6)

### 2. Primer contacto con el DataFrame
- [2.1 Dimensiones y primeras filas](#sec-2-1)
- [2.2 Estructura y tipos de dato](#sec-2-2)
- [2.3 Estadísticos descriptivos](#sec-2-3)
- [2.4 Variable objetivo y variables predictoras](#sec-2-4)

### 3. Análisis Exploratorio de Datos (EDA)
- [3.1 Librerías de gráficos y estilo](#sec-3-1)
- [3.2 Distribución de la variable objetivo](#sec-3-2)
- [3.3 Estadísticos ampliados](#sec-3-3)
- [3.4 Histogramas](#sec-3-4)
- [3.5 Boxplots](#sec-3-5)
- [3.6 Densidad por clase](#sec-3-6)
- [3.7 Relaciones entre variables](#sec-3-7)
- [3.8 Mapa de calor de correlaciones](#sec-3-8)
- [3.9 Un gráfico interactivo con Plotly](#sec-3-9)

### 4. Limpieza y consolidación
- [4.1 Valores faltantes](#sec-4-1)
- [4.2 Registros duplicados](#sec-4-2)
- [4.3 Tipos de dato](#sec-4-3)
- [4.4 Rangos y consistencia](#sec-4-4)
- [4.5 Outliers: detección y decisión](#sec-4-5)
- [4.6 Resumen de la auditoría de calidad](#sec-4-6)

### 5. Transformaciones básicas
- [5.1 Codificación de variables categóricas](#sec-5-1)
- [5.2 Escalado de variables numéricas](#sec-5-2)
- [5.3 Creación o eliminación de variables](#sec-5-3)

### 6. Selección de variables relevantes
- [6.1 Tres criterios de relevancia](#sec-6-1)
- [6.2 La decisión, y por qué se toma acá y no antes](#sec-6-2)

### 7. División en entrenamiento y prueba
- [7.1 Por qué hay que dividir](#sec-7-1)
- [7.2 La división 80/20 estratificada](#sec-7-2)
- [7.3 Recién ahora: aplicar el escalado sin fuga de información](#sec-7-3)

### 8. Exportar los resultados
- [8.1 Los archivos que deja el cuaderno](#sec-8-1)
- [8.2 Cómo llevárselos](#sec-8-2)

### Cierre
- [Resumen y observaciones finales](#sec-cierre)

---
# Parte 0 — Preliminares

<a id="sec-0-1"></a>
## 0.1 El entorno de trabajo: Google Colab

Google Colab es un entorno de cuadernos Jupyter que corre en los servidores de Google. No hay que
instalar nada: alcanza con el navegador y una cuenta de Google. Cada cuaderno se ejecuta en una
máquina virtual con Python y las librerías de ciencia de datos ya instaladas.

Cuatro cosas conviene tener claras antes de empezar:

**1. El cuaderno tiene estado.** Las variables definidas en una celda siguen existiendo en las
siguientes. Por eso el orden de ejecución importa: hay que correr las celdas de arriba hacia
abajo. Si una celda falla diciendo que una variable no existe, casi siempre es porque se salteó
una celda anterior.

**2. La máquina es temporal.** Si la sesión se desconecta —por inactividad o por cerrar la
pestaña un rato largo— se pierde todo lo que esté guardado en el disco de esa máquina, incluidos
los archivos subidos a mano. El cuaderno en sí no se pierde: vive en tu Google Drive.

**3. Hay dos tipos de celda.** Las de código, que ejecutan Python, y las de texto (Markdown),
que explican. Este cuaderno pone una de texto antes de cada una de código.

**4. Las librerías ya están.** Pandas, NumPy, Matplotlib, Seaborn, Plotly y scikit-learn vienen
preinstaladas.

`openpyxl` no se importa nunca directamente: es lo que Pandas usa por debajo para leer `.xlsx`.

<a id="sec-0-2"></a>
## 0.2 Cargar el dataset

El archivo con los datos no está adentro de Colab: hay que llevarlo hasta ahí. Hay cuatro
maneras, y todas terminan en lo mismo: una variable `RUTA_ARCHIVO` con la ubicación del archivo.

| | Opción | Cuándo conviene |
|---|---|---|
| **A** | **GitHub** *(activa)* | El archivo está en un repositorio público. Quien abre el cuaderno no tiene que descargar ni subir nada |
| **B** | Subir a mano | Prueba rápida en Colab con un archivo propio. Se pierde al reiniciarse la sesión |
| **C** | Google Drive | Vas a trabajar varios días con el mismo archivo |
| **D** | Ruta local | Corrés el cuaderno en tu máquina, con Jupyter o VS Code |

**Formatos.** Pandas lee prácticamente cualquier formato tabular, y elige la función según la
extensión: `read_csv()` para `.csv` y `.txt`, `read_excel()` para `.xlsx` y `.xls`,
`read_json()`, `read_parquet()`. Las dos primeras cubren el 95% de los casos, y **las dos
funcionan igual desde una URL que desde un archivo local**.

Los parámetros que más problemas evitan: `sheet_name` (qué hoja del Excel), `sep` (el separador
del CSV, a veces `;`), `decimal` (a veces `,`), `encoding` (si los acentos salen rotos) y
`skiprows` (si la tabla no empieza en la primera fila).

<a id="sec-0-2-1"></a>
### 0.2.1 Las cuatro opciones

La celda de abajo tiene las cuatro. **La A está activa**; las otras tres están comentadas. Para
cambiar de opción, comentá la A y descomentá la que necesites.

Para usar la opción A con tu propio repositorio solo hay que cambiar dos líneas: `USUARIO` y
`REPO`. La URL se arma sola. El repositorio tiene que ser **público**; si es privado, la
dirección `raw` pide autenticación y no funciona.

In [ ]:
# =====================================================================
#  ELEGÍ UNA OPCIÓN — la A está activa, las demás comentadas
# =====================================================================

NOMBRE_ARCHIVO = "doc-topics_K07_it1500_oi0_ob0_a1_b0.01_s12345.xlsx"

# --- A) GitHub  (ACTIVA) ---------------------------------------------
USUARIO = "TU_USUARIO"        # <-- cambiar
REPO    = "TU_REPO"           # <-- cambiar
RAMA    = "main"              # GitHub usa 'main' por defecto desde 2020

RUTA_ARCHIVO = f"https://raw.githubusercontent.com/{USUARIO}/{REPO}/{RAMA}/{NOMBRE_ARCHIVO}"

# --- B) Subir el archivo a mano  (solo Colab) ------------------------
# from google.colab import files
# RUTA_ARCHIVO = list(files.upload().keys())[0]

# --- C) Google Drive  (solo Colab) -----------------------------------
#     Truco: una vez montado, clic derecho sobre el archivo en el panel
#     de la izquierda -> "Copiar ruta", y pegás la ruta exacta.
# from google.colab import drive
# drive.mount("/content/drive")
# RUTA_ARCHIVO = f"/content/drive/MyDrive/datasets/{NOMBRE_ARCHIVO}"

# --- D) Archivo local  (Jupyter / VS Code) ---------------------------
# RUTA_ARCHIVO = NOMBRE_ARCHIVO
# RUTA_ARCHIVO = r"C:\Users\usuario\Documentos\datos.xlsx"   # ruta absoluta en Windows

print(f"Origen configurado : {RUTA_ARCHIVO}")
if "TU_USUARIO" in str(RUTA_ARCHIVO):
    print()
    print("[AVISO] La opción A todavía tiene los valores de ejemplo.")
    print("        Cambiá USUARIO y REPO, o descomentá otra opción.")
    print("        Mientras tanto, la celda siguiente va a usar el archivo local.")

<a id="sec-0-2-2"></a>
### 0.2.2 Leer el archivo

Importamos las dos librerías de esta etapa y leemos.

- **NumPy** (`np`) — cálculo numérico sobre arreglos; es la base de todo lo demás.
- **Pandas** (`pd`) — datos tabulares. Sus dos estructuras son la `Series` (una columna) y el
  `DataFrame` (una tabla).

La función `leer_datos()` elige `read_csv` o `read_excel` según la extensión, así que el cuaderno
funciona igual con un `.xlsx` que con un `.csv` sin tocar nada.

Y si la ruta configurada falla —por ejemplo, porque la opción A todavía tiene los valores de
ejemplo— la celda avisa y reintenta con el archivo local antes de darse por vencida.

In [ ]:
# --- Librerías de manipulación de datos ---
import numpy as np
import pandas as pd


def leer_datos(ruta):
    # Elige la función de lectura según la extensión. Funciona igual
    # con una ruta local que con una URL.
    if str(ruta).lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(ruta)
    return pd.read_csv(ruta)


# --- Lectura, con reintento local si la ruta configurada falla ---
try:
    df = leer_datos(RUTA_ARCHIVO)
except Exception as error:
    print(f"[AVISO] No se pudo leer: {RUTA_ARCHIVO}")
    print(f"        {type(error).__name__}: {str(error)[:110]}")
    print(f"        Reintentando con el archivo local '{NOMBRE_ARCHIVO}'...")
    print()
    df = leer_datos(NOMBRE_ARCHIVO)
    RUTA_ARCHIVO = NOMBRE_ARCHIVO

print("=" * 70)
print("  LECTURA")
print("=" * 70)
print(f"  NumPy / Pandas : {np.__version__} / {pd.__version__}")
print(f"  Archivo        : {RUTA_ARCHIVO}")
print(f"  Dimensiones    : {df.shape[0]} filas x {df.shape[1]} columnas")
print()

if df.empty:
    print("  [ERROR] El DataFrame quedó vacío. Revisá la hoja (sheet_name) o skiprows.")
elif df.shape[1] == 1:
    print("  [AVISO] Una sola columna: casi siempre es el separador equivocado.")
    print("          Probá pd.read_csv(..., sep=';')")
else:
    print("  [OK] El archivo se leyó correctamente.")

print()
print(f"  Columnas: {list(df.columns)}")
print()
display(df.head())

<a id="sec-0-2-3"></a>
### 0.2.3 Si la lectura sale mal

| Síntoma | Causa probable | Solución |
|---|---|---|
| `FileNotFoundError` / error 404 | Ruta o URL mal escrita; el repo es privado | Verificar la ruta; hacer público el repositorio |
| Una sola columna con todo adentro | Separador equivocado | `sep=";"` o `sep="\t"` |
| Números leídos como texto | Decimal con coma | `decimal=","` |
| Acentos rotos (`Ã³`, `Ã±`) | Codificación equivocada | `encoding="latin-1"` o `"utf-8-sig"` |
| Encabezados `Unnamed: 0`, `Unnamed: 1`… | La tabla no empieza en la primera fila | `skiprows=3` |
| Faltan filas o columnas | Se leyó la hoja equivocada | `sheet_name="Datos"` |
| `Missing optional dependency 'openpyxl'` | Falta la librería que lee `.xlsx` | `pip install openpyxl` |

La celda siguiente inspecciona el archivo ya cargado: qué hojas tiene, si es un Excel, y qué tipo
dedujo Pandas para cada columna.

Un recordatorio para leer esa salida: `object` significa texto —o mezcla de texto y números—,
`int64` enteros y `float64` decimales. **Si una columna que debería ser numérica aparece como
`object`, hay algún valor no numérico escondido adentro.** Lo revisamos en el punto 4.3.

In [ ]:
print("=" * 70)
print("  DIAGNÓSTICO DEL ARCHIVO")
print("=" * 70)

try:
    hojas = pd.ExcelFile(RUTA_ARCHIVO).sheet_names
    print(f"  Hojas del Excel : {hojas}")
    print(f"  Hoja leída      : {hojas[0]}  (la primera, que es el valor por defecto)")
except Exception:
    print("  No es un Excel, o no se pudo inspeccionar como tal.")

print()
print("  Tipo que Pandas dedujo para cada columna:")
for columna, tipo in df.dtypes.items():
    print(f"    {str(columna):<14} -> {tipo}")

<a id="sec-0-3"></a>
## 0.3 Configuración global

Todo lo que se podría querer cambiar más adelante queda junto acá arriba, con nombre propio, en
lugar de repartido por el cuaderno como número suelto.

In [ ]:
# --- Estructura del dataset ---
COLUMNA_OBJETIVO = "TAG"      # la variable a predecir; siempre la primera columna

# --- Umbrales de las auditorías ---
UMBRAL_Z         = 3.0        # |z| por encima de esto: candidato a outlier
FACTOR_IQR       = 1.5        # regla de Tukey
UMBRAL_CORR_ALTA = 0.80       # correlación que consideramos preocupante

# --- Salida del cuaderno ---
ARCHIVO_LIMPIO   = "dataset_limpio.csv"     # el dataset auditado (punto 8)
ARCHIVO_ENTRENA  = "entrenamiento.csv"      # el 80% de entrenamiento (punto 8)
ARCHIVO_PRUEBA   = "prueba.csv"             # el 20% de prueba (punto 8)

# --- Modelado ---
SEMILLA          = 42     # fija la aleatoriedad: el cuaderno da siempre lo mismo
PROPORCION_PRUEBA = 0.20  # 20% para prueba, 80% para entrenamiento
CLASE_POSITIVA   = "AFI"  # cuál de las dos clases se codifica como 1

# --- Formato de la salida por pantalla ---
import warnings
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
warnings.filterwarnings("ignore")

print("Configuración cargada.")
print(f"  Variable objetivo   : {COLUMNA_OBJETIVO}")
print(f"  Factor IQR          : {FACTOR_IQR}")
print(f"  Umbral z            : {UMBRAL_Z}")
print(f"  Archivo de salida   : {ARCHIVO_LIMPIO}")
print(f"  Semilla             : {SEMILLA}")
print(f"  Proporción de prueba: {PROPORCION_PRUEBA:.0%}")
print(f"  Clase positiva      : {CLASE_POSITIVA}")

---
# 1. Selección del dataset

<a id="sec-1-1"></a>
## 1.1 Origen de los datos

Los datos provienen del debate parlamentario argentino sobre la **Interrupción Voluntaria del
Embarazo (IVE)**, sancionada después como
[Ley 27.610](https://servicios.infoleg.gob.ar/infolegInternet/verNorma.do?id=346231). Se trabajó
únicamente con la **Cámara de Diputados**.

Las fuentes son oficiales:

| Fuente | Para qué se usó |
|---|---|
| [Diario de Sesiones (versión taquigráfica)](https://www3.hcdn.gob.ar/dependencias/dtaquigrafos/diarios/periodo-138/diario_2020121017.pdf) | Paginación, apéndice, inserciones |
| [Transcripción HTML de la sesión](https://www4.hcdn.gob.ar/sesionesxml/provisorias/138-17.htm) | Texto continuo de las intervenciones |
| [Votación nominal — Acta 1, O.D. 352](https://votaciones.hcdn.gob.ar/votacion/4077) | Nombre, bloque, provincia y voto |
| [Orden del Día Nº 352](https://www4.hcdn.gob.ar/dependencias/dcomisiones/periodo-138/138-352.pdf) | Delimitar el asunto tratado |

De los 257 diputados, no todos intervinieron. Tras excluir las abstenciones el corpus quedó en
**171 discursos: 95 afirmativos (AFI) y 76 negativos (NEG)**.

Antes de procesar, cada discurso se llevó a minúsculas, se le quitaron números y símbolos, y se le
aplicó una lista de **205 stopwords** construida específicamente para este corpus.

<a id="sec-1-2"></a>
## 1.2 De texto a números: el modelo de tópicos

Los 171 discursos se procesaron con **[MALLET](https://github.com/mimno/Mallet)**, herramienta de
procesamiento de lenguaje natural desarrollada en la Universidad de Massachusetts. Se usó su
implementación de **[LDA](https://www.jmlr.org/papers/v3/blei03a.html)** (*Latent Dirichlet
Allocation*).

Idea brevísima de LDA: **cada documento es una mezcla de tópicos, y cada tópico es una mezcla de
palabras**. El modelo no sabe de antemano cuáles son esos tópicos; los descubre observando qué
palabras tienden a aparecer juntas.

La **distribución de Dirichlet** es la que gobierna esas dos mezclas, a través de dos parámetros:

- **α (alpha)** — cuán concentrados están los tópicos dentro de un documento. Alpha bajo produce
  discursos dominados por un solo tema; alpha alto, discursos repartidos entre varios.
- **β (beta)** — cuán específico es el vocabulario de cada tópico. Beta bajo da tópicos con pocas
  palabras muy distintivas; beta alto los abre a vocabulario general.

El algoritmo arranca asignando cada palabra a un tópico al azar y va refinando esas asignaciones
iteración tras iteración —mediante muestreo de Gibbs— hasta que se estabilizan.

La salida es lo que constituye este dataset: **cada discurso queda reducido a un vector de 7
proporciones que suman 1**, una por tópico. Eso explica dos cosas que van a aparecer en el
análisis: por qué las filas suman exactamente 1, y por qué las correlaciones entre columnas son
casi todas negativas.

<a id="sec-1-3"></a>
## 1.3 La búsqueda de la mejor separación

El modelo de tópicos no se parametrizó buscando tópicos interpretables (que no nos interesan),
sino **tópicos que separaran bien las dos posiciones de voto**.

Se probaron sistemáticamente cantidades de tópicos (5, 7, 10), iteraciones, alpha, beta, y la
optimización automática de hiperparámetros. Cada configuración se evaluó con un informe de
separabilidad que mide, con un modelo de referencia fijo, cuán bien se distinguen las clases.

La configuración que quedó fija es: **K=7 tópicos, 1500 iteraciones, α=1, β=0,01, sin optimización
automática, semilla 12345**.

Resultado de esa configuración:

- Separabilidad global: **AUC 0,9660** (IC 95 %: 0,9627–0,9693).
- 5 de los 7 tópicos aportan señal; 1 no distingue las clases.
- Con 4 tópicos ya se alcanza prácticamente el techo.
- Un modelo lineal y uno no lineal rinden casi igual, lo que sugiere que la separación es de
  naturaleza lineal.

<a id="sec-1-4"></a>
## 1.4 El problema que aborda

Hasta acá hay una representación numérica de cada discurso. La pregunta que sigue es la que da
sentido a estos dos cuadernos:

> **¿Se puede predecir cómo votó un diputado a partir de cómo habló?**

Formalmente: dado el vector de 7 proporciones de un discurso, predecir si el voto fue **AFI** o
**NEG**. Es un problema de **clasificación binaria supervisada**, y se resuelve con un **segundo
modelo de aprendizaje automático** encadenado al primero:

```
discurso  →  [MALLET / LDA]  →  7 proporciones  →  [clasificador]  →  AFI / NEG
              modelo 1                                  modelo 2
```

Este cuaderno prepara los datos para ese segundo modelo. La Parte 2 lo entrena y lo evalúa.

El interés no es adivinar votos ya conocidos y públicos, sino medir **cuánta información sobre la
posición está contenida en la forma de argumentar**.

<a id="sec-1-5"></a>
## 1.5 Justificación de la elección

- **Es un problema real con conocimiento del dominio disponible**, lo que permite juzgar si los
  resultados tienen sentido y no solo si las métricas dan bien.
- **Las clases están razonablemente balanceadas** (95 contra 76, razón 1,25 a 1).
- **Es tabular, de tamaño moderado y completo**, así que el esfuerzo se concentra en aplicar bien
  cada etapa del proceso.
- **Las siete variables son todas numéricas y están en la misma escala**, lo que simplifica el
  preprocesamiento.
- **La trazabilidad está documentada de punta a punta**, desde las fuentes oficiales hasta los
  parámetros del modelo de tópicos.

**Limitaciones, dichas de entrada:**

| Limitación | Por qué importa |
|---|---|
| Solo 171 filas | Las estimaciones tienen margen de error amplio; van a obligar a usar validación cruzada |
| La versión taquigráfica no es una transcripción literal | Es la documentación oficial, con su propia edición |

<a id="sec-1-6"></a>
## 1.6 Ficha técnica

Antes de cualquier análisis, una ficha con los datos duros del archivo. La celda la arma sola, así
que si el día de mañana el dataset cambia, la ficha se actualiza sin tocar nada.

Es lo que se pega en la primera página de un informe: tamaño, variables, clases y peso.

In [ ]:
print("=" * 72)
print("  FICHA TÉCNICA DEL DATASET")
print("=" * 72)
print(f"  Archivo                : {RUTA_ARCHIVO}")
print(f"  Formato                : {'Excel (.xlsx)' if str(RUTA_ARCHIVO).lower().endswith(('.xlsx', '.xls')) else 'CSV / texto'}")
print(f"  Observaciones (filas)  : {df.shape[0]}")
print(f"  Variables (columnas)   : {df.shape[1]}")
print(f"  Peso en memoria        : {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
print()
print(f"  Variable objetivo      : {COLUMNA_OBJETIVO}")
print(f"  Tipo de problema       : clasificación binaria supervisada")
print(f"  Clases                 : {sorted(df[COLUMNA_OBJETIVO].astype(str).unique())}")
print()

reparto = df[COLUMNA_OBJETIVO].value_counts()
for clase, cantidad in reparto.items():
    print(f"      {clase:>5} : {cantidad:>4} casos  ({cantidad / len(df):.1%})")
print(f"  Razón mayoritaria/minoritaria : {reparto.max() / reparto.min():.2f} a 1")
print()

numericas = df.select_dtypes(include="number").columns.tolist()
categoricas = [c for c in df.columns if c not in numericas]
print(f"  Variables numéricas    : {len(numericas)}")
print(f"  Variables categóricas  : {len(categoricas)}  ->  {categoricas}")
print(f"  Valores faltantes      : {int(df.isna().sum().sum())}")
print(f"  Filas duplicadas       : {int(df.duplicated().sum())}")
print("=" * 72)

---
# 2. Primer contacto con el DataFrame

Antes de graficar nada, cuatro preguntas básicas:

1. **¿Qué tamaño tiene?** Cuántas filas (observaciones) y cuántas columnas (variables).
2. **¿Cómo son  los datos?** Las primeras y las últimas filas, para confirmar que el archivo se leyó bien.
3. **¿De qué tipo es cada columna?** Una columna numérica leída como texto es uno de los errores más frecuentes y más silenciosos.
4. **¿Cómo se distribuyen los valores?** Media, desvío, mínimo, máximo y cuartiles.

<a id="sec-2-1"></a>
## 2.1 Dimensiones y primeras filas

In [ ]:
print("=" * 70)
print("  DIMENSIONES")
print("=" * 70)
print(f"  Filas    : {df.shape[0]}")
print(f"  Columnas : {df.shape[1]}")
print(f"  Celdas   : {df.size}")
print(f"  Memoria  : {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

print()
print("=" * 70)
print("  PRIMERAS 5 FILAS")
print("=" * 70)
display(df.head())

print("=" * 70)
print("  ÚLTIMAS 5 FILAS")
print("=" * 70)
display(df.tail())

<a id="sec-2-2"></a>
## 2.2 Estructura y tipos de dato

`info()` es probablemente el método más útil de Pandas para un primer diagnóstico. En una sola
salida muestra, columna por columna: el nombre, cuántos valores **no nulos** tiene y de qué tipo
es.

Cómo leerlo: si el "Non-Null Count" de una columna es menor que la cantidad total de filas, esa
columna tiene valores faltantes. Y si una columna que debería ser numérica aparece como `object`,
Pandas la leyó como texto.

In [ ]:
print("=" * 70)
print("  ESTRUCTURA (info)")
print("=" * 70)
df.info()

print()
print("=" * 70)
print("  RESUMEN POR COLUMNA")
print("=" * 70)
resumen_tipos = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "no_nulos": df.notna().sum(),
    "nulos": df.isna().sum(),
    "valores_unicos": df.nunique(),
})
display(resumen_tipos)

<a id="sec-2-3"></a>
## 2.3 Estadísticos descriptivos

`describe()` calcula los estadísticos de las columnas numéricas: cantidad, media, desvío estándar,
mínimo, los tres cuartiles y máximo.

Lo transponemos con `.T` porque con muchas columnas se lee mejor en vertical: cada fila pasa a ser
una variable y cada columna un estadístico.

Qué mirar:

- **`min` y `max`** — ¿los valores están en el rango que esperábamos? Un negativo en una variable
  que debería ser siempre positiva es una bandera roja.
- **`std` igual a 0** — la columna es constante y no aporta nada.
- **La distancia entre `75%` y `max`** — si es enorme comparada con la que hay entre `25%` y
  `75%`, hay valores extremos hacia arriba.
- **`mean` contra `50%`** — si son muy distintas, la distribución es asimétrica.

In [ ]:
print("=" * 70)
print("  ESTADÍSTICOS DESCRIPTIVOS (columnas numéricas)")
print("=" * 70)
display(df.describe().T)

print("=" * 70)
print("  COLUMNAS NO NUMÉRICAS")
print("=" * 70)
no_numericas = df.select_dtypes(exclude=[np.number])
if no_numericas.shape[1] == 0:
    print("  No hay columnas no numéricas.")
else:
    display(no_numericas.describe().T)

<a id="sec-2-4"></a>
## 2.4 Variable objetivo y variables predictoras

Separamos dos grupos de columnas con los que vamos a trabajar:

- **La variable objetivo** (`TAG`) — lo que en la Parte 2 vamos a querer predecir.
- **Las variables predictoras** (o *features*) — las columnas numéricas a partir de las cuales
  queremos predecirla.

Los guardamos en variables con nombre para no volver a escribir la lista nunca más. Notá que la
lista se arma sola: si el día de mañana el archivo tuviera nueve columnas numéricas en lugar de
siete, esta celda las tomaría todas sin tocar nada del resto del cuaderno.

In [ ]:
objetivo = COLUMNA_OBJETIVO
predictoras = [c for c in df.columns
               if c != objetivo and pd.api.types.is_numeric_dtype(df[c])]

print(f"Variable objetivo     : {objetivo}")
print(f"Variables predictoras : {len(predictoras)}")
for c in predictoras:
    print(f"    - {c}")

ignoradas = [c for c in df.columns if c != objetivo and c not in predictoras]
if ignoradas:
    print()
    print(f"Columnas que no son ni objetivo ni predictora numérica: {ignoradas}")

---
# 3. Análisis Exploratorio de Datos (EDA)

El EDA es la etapa en la que tratamos de **entender** los datos antes de modelarlos. No se trata
de hacer gráficos por hacer: cada uno responde una pregunta concreta.

El orden va de lo simple a lo complejo:

1. **Una variable a la vez** — ¿cómo se distribuye cada columna por separado? (3.3 a 3.5)
2. **Cada variable contra el objetivo** — ¿cuáles separan las clases y cuáles no? (3.6)
3. **Las variables entre sí** — ¿hay columnas que dicen lo mismo? (3.7 y 3.8)

Empezamos por la variable objetivo, porque todo lo demás se lee en función de ella.

<a id="sec-3-1"></a>
## 3.1 Librerías de gráficos y estilo

Ahora que vamos a graficar, importamos las librerías de visualización. Son tres y cada una ocupa
un lugar distinto:

| Librería | Qué aporta |
|---|---|
| **Matplotlib** | La librería clásica, de 2003. Controla cada detalle del gráfico: colores, ejes, leyendas, escalas. Todo lo demás está construido encima |
| **Seaborn** | Construida sobre Matplotlib. Menos código y mejor estética por defecto para gráficos estadísticos: densidades, boxplots, mapas de calor, pairplots |
| **Plotly** | Gráficos **interactivos**: zoom, desplazamiento, valores al pasar el cursor. La usamos una sola vez, en 3.9 |

Aprovechamos la celda para fijar el estilo de todos los gráficos del cuaderno —tamaño, resolución,
tipografía— y los colores de cada clase. Que el color de `AFI` sea siempre el mismo en todos los
gráficos no es cosmético: hace que se puedan comparar de un vistazo.

In [ ]:
# --- Librerías de gráficos ---
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# --- Utilidad para recorrer pares de columnas (la usa 3.8) ---
from itertools import combinations

# --- Estilo global ---
sns.set_theme(style="whitegrid", palette="deep")
mpl.rcParams["figure.figsize"] = (10, 5)
mpl.rcParams["figure.dpi"] = 110
mpl.rcParams["axes.titlesize"] = 12
mpl.rcParams["axes.titleweight"] = "bold"
mpl.rcParams["font.size"] = 10

# --- Un color fijo por clase, para todo el cuaderno ---
COLOR_CLASE = {"AFI": "#1f77b4", "NEG": "#d62728"}

print(f"Matplotlib : {mpl.__version__}")
print(f"Seaborn    : {sns.__version__}")
print()
print("Estilo configurado. Colores por clase:")
for clase, color in COLOR_CLASE.items():
    print(f"    {clase} -> {color}")

<a id="sec-3-2"></a>
## 3.2 Distribución de la variable objetivo

Lo primero que hay que saber de un problema de clasificación es **cuántos casos hay de cada
clase**. Ese reparto condiciona todo lo que viene después:

- Si una clase es mucho más frecuente que la otra, el dataset está **desbalanceado**, y el
  accuracy deja de ser una métrica confiable: un modelo que responda siempre la clase mayoritaria
  puede tener un accuracy altísimo y ser completamente inútil.
- El reparto define la **línea de base**: la proporción de la clase mayoritaria es lo que
  acertaría el modelo más tonto posible. Cualquier modelo que construyamos en la Parte 2 tiene que
  superar ese número, o no sirve para nada.

In [ ]:
conteo = df[objetivo].value_counts()
proporcion = df[objetivo].value_counts(normalize=True)

print("=" * 70)
print("  REPARTO DE LA VARIABLE OBJETIVO")
print("=" * 70)
for clase in conteo.index:
    barra = "#" * int(proporcion[clase] * 50)
    print(f"  {clase:>5} : {conteo[clase]:>4} casos  ({proporcion[clase]:6.2%})  {barra}")

linea_base = proporcion.max()
clase_mayoritaria = proporcion.idxmax()
razon = conteo.max() / conteo.min()

print()
print(f"  Total de casos    : {len(df)}")
print(f"  Clase mayoritaria : {clase_mayoritaria}")
print(f"  Razón mayor/menor : {razon:.2f} a 1")
print(f"  LÍNEA DE BASE     : {linea_base:.2%}")
print()
print(f"  Todo modelo que construyamos tendrá que superar el {linea_base:.1%} de accuracy,")
print(f"  porque ese es el resultado de responder siempre '{clase_mayoritaria}'.")
print()
if razon <= 1.5:
    print("  DIAGNÓSTICO: clases razonablemente balanceadas.")
elif razon <= 3:
    print("  DIAGNÓSTICO: desbalance leve. Conviene mirar F1 y balanced accuracy,")
    print("               no solo el accuracy.")
else:
    print("  DIAGNÓSTICO: desbalance fuerte. El accuracy es engañoso; hay que")
    print("               apoyarse en F1, balanced accuracy y la matriz de confusión.")
print()
fig, ejes = plt.subplots(1, 2, figsize=(12, 4.2))

colores = [COLOR_CLASE.get(c, "#888888") for c in conteo.index]
ejes[0].bar(conteo.index.astype(str), conteo.values, color=colores,
            edgecolor="black", linewidth=0.6)
for i, v in enumerate(conteo.values):
    ejes[0].text(i, v + 1.5, str(v), ha="center", fontweight="bold")
ejes[0].set_title("Cantidad de casos por clase")
ejes[0].set_ylabel("casos")
ejes[0].set_ylim(0, conteo.max() * 1.15)

ejes[1].pie(conteo.values, labels=conteo.index.astype(str), colors=colores,
            autopct="%1.1f%%", startangle=90,
            wedgeprops={"edgecolor": "white", "linewidth": 1.5})
ejes[1].set_title("Proporción de cada clase")

plt.tight_layout()
plt.show()

<a id="sec-3-3"></a>
## 3.3 Estadísticos ampliados

A lo que dio `describe()` le agregamos tres medidas que ayudan a anticipar problemas:

- **Asimetría (*skewness*)** — mide cuán inclinada está la distribución. Cero es simétrica; un
  valor positivo grande significa una cola larga hacia la derecha (muchos valores chicos y unos
  pocos muy grandes). Por encima de 1 en valor absoluto la asimetría ya es marcada.
- **Curtosis** — mide cuán pesadas son las colas. Valores altos indican más valores extremos de los
  que tendría una distribución normal.
- **Coeficiente de variación** — el desvío dividido por la media. Permite comparar la dispersión de
  variables que están en escalas distintas, cosa que el desvío solo no permite.

La razón práctica de mirar esto ahora es que la asimetría y las colas pesadas son exactamente lo
que va a aparecer después como "outliers" en el punto 3.5.

In [ ]:
descripcion = df[predictoras].describe().T
descripcion["asimetria"] = df[predictoras].skew()
descripcion["curtosis"] = df[predictoras].kurtosis()
descripcion["coef_variacion"] = df[predictoras].std() / df[predictoras].mean()
descripcion["rango"] = descripcion["max"] - descripcion["min"]
descripcion["rango_intercuartil"] = descripcion["75%"] - descripcion["25%"]

print("=" * 70)
print("  ESTADÍSTICOS AMPLIADOS")
print("=" * 70)
display(descripcion[["mean", "50%", "std", "min", "max", "rango",
                     "rango_intercuartil", "asimetria", "curtosis", "coef_variacion"]])

print()
print("  Lectura rápida:")
for c in predictoras:
    asim = descripcion.loc[c, "asimetria"]
    marca = "  <-- asimetría marcada" if abs(asim) > 1 else ""
    print(f"    {c:<12} media={descripcion.loc[c, 'mean']:.4f}  "
          f"mediana={descripcion.loc[c, '50%']:.4f}  asimetría={asim:+.2f}{marca}")

<a id="sec-3-4"></a>
## 3.4 Histogramas

El histograma muestra la **forma** de la distribución de una variable: agrupa los valores en
intervalos y dibuja una barra con la cantidad de casos de cada uno.

Es la manera más directa de ver si una variable es simétrica o está inclinada hacia un lado, si
tiene uno o varios picos, y si hay valores sueltos muy alejados del resto.

Le superponemos la curva de densidad (`kde=True`), que es la versión continua del mismo histograma
y ayuda a ver la forma sin depender de dónde caen los cortes de los intervalos. La línea vertical
marca la media.

In [ ]:
n = len(predictoras)
filas = int(np.ceil(n / 3))
fig, ejes = plt.subplots(filas, 3, figsize=(14, 3.2 * filas))
ejes = np.array(ejes).ravel()

for i, col in enumerate(predictoras):
    sns.histplot(df[col], bins=25, kde=True, ax=ejes[i],
                 color="#4c72b0", edgecolor="white", linewidth=0.4)
    ejes[i].axvline(df[col].mean(), color="#d62728", linestyle="--", linewidth=1.4,
                    label=f"media = {df[col].mean():.3f}")
    ejes[i].set_title(col)
    ejes[i].set_xlabel("")
    ejes[i].legend(fontsize=8)

for j in range(n, len(ejes)):
    ejes[j].axis("off")

fig.suptitle("Distribución de cada variable predictora", fontsize=13, fontweight="bold", y=1.00)
plt.tight_layout()
plt.show()

<a id="sec-3-5"></a>
## 3.5 Boxplots

El boxplot (o diagrama de caja) resume una distribución en cinco números:

- La **caja** va del primer cuartil (Q1, el 25% de los datos está por debajo) al tercero (Q3, el
  75%). O sea, contiene la mitad central de los datos.
- La **línea dentro de la caja** es la mediana.
- Los **bigotes** llegan hasta el dato más alejado que todavía está dentro de
  `Q1 - 1,5 x IQR` y `Q3 + 1,5 x IQR`, donde IQR es el alto de la caja.
- Los **puntos sueltos** más allá de los bigotes son los candidatos a outlier según la regla de
  Tukey. Los analizamos en detalle en 3.5.

La gran ventaja del boxplot es que permite comparar muchas variables en un solo gráfico. Lo
dibujamos dos veces: una con todas en la misma escala, para comparar niveles, y otra con cada
variable reescalada, para ver su forma sin que las más grandes aplasten a las más chicas.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 4.6))

datos_largos = df[predictoras].melt(var_name="variable", value_name="valor")
sns.boxplot(data=datos_largos, x="variable", y="valor", ax=ejes[0],
            color="#4c72b0", fliersize=3)
ejes[0].set_title("Todas las variables en la misma escala")
ejes[0].tick_params(axis="x", rotation=45)
ejes[0].set_xlabel("")

normalizado = (df[predictoras] - df[predictoras].min()) / (df[predictoras].max() - df[predictoras].min())
datos_norm = normalizado.melt(var_name="variable", value_name="valor")
sns.boxplot(data=datos_norm, x="variable", y="valor", ax=ejes[1],
            color="#55a868", fliersize=3)
ejes[1].set_title("Cada variable reescalada a 0-1 (solo para comparar formas)")
ejes[1].tick_params(axis="x", rotation=45)
ejes[1].set_xlabel("")

plt.tight_layout()
plt.show()

print("  Ojo: el reescalado del gráfico de la derecha es SOLO para mirar. No")
print("  modifica el DataFrame. El escalado de verdad, el que usan los modelos,")
print("  se hace en el punto 5, y se APLICA recién en el 7.3, después de dividir.")

<a id="sec-3-6"></a>
## 3.6 Densidad por clase

Este es, para un problema de clasificación, **el gráfico más informativo de todo el EDA**.

Para cada variable dibujamos dos curvas de densidad: una con los casos de la clase `AFI` y otra
con los de `NEG`. La pregunta que responde es directa: **¿esta variable ayuda a distinguir las
clases?**

- Si las dos curvas están **superpuestas**, la variable no aporta nada por sí sola: conociendo su
  valor no sabemos a qué clase pertenece el caso.
- Si están **corridas una respecto de la otra**, la variable separa. Cuanto menos se pisan, mejor.

Para ordenar los gráficos de mayor a menor poder de separación usamos la **diferencia
estandarizada de medias**, conocida como *d de Cohen*: la diferencia entre las medias de las dos
clases, dividida por el desvío combinado. Al dividir por el desvío, el resultado no depende de la
escala de la variable, y por eso se pueden comparar variables distintas entre sí. Como referencia
habitual: 0,2 es un efecto chico, 0,5 mediano y 0,8 grande.

Una aclaración importante: esto mide el poder de separación de **cada variable por separado**. Un
modelo combina todas a la vez y puede aprovechar información que ninguna tiene sola. Que una
variable separe poco acá no es motivo suficiente para descartarla.

In [ ]:
def d_de_cohen(serie, etiquetas, clase_a, clase_b):
    a = serie[etiquetas == clase_a].to_numpy()
    b = serie[etiquetas == clase_b].to_numpy()
    na, nb = len(a), len(b)
    s = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    if s == 0:
        return 0.0
    return (a.mean() - b.mean()) / s


def etiqueta_efecto(m):
    if m < 0.2:
        return "despreciable"
    if m < 0.5:
        return "chico"
    if m < 0.8:
        return "mediano"
    return "grande"


clases = sorted(df[objetivo].unique())
clase_a, clase_b = clases[0], clases[1]

separacion = pd.DataFrame({
    "variable": predictoras,
    "d_cohen": [d_de_cohen(df[c], df[objetivo], clase_a, clase_b) for c in predictoras],
})
separacion["magnitud"] = separacion["d_cohen"].abs()
separacion[f"media_{clase_a}"] = [df.loc[df[objetivo] == clase_a, c].mean() for c in predictoras]
separacion[f"media_{clase_b}"] = [df.loc[df[objetivo] == clase_b, c].mean() for c in predictoras]
separacion["inclina_hacia"] = np.where(separacion["d_cohen"] > 0, clase_a, clase_b)
separacion["efecto"] = separacion["magnitud"].map(etiqueta_efecto)
separacion = separacion.sort_values("magnitud", ascending=False).reset_index(drop=True)

print("=" * 78)
print("  PODER DE SEPARACIÓN DE CADA VARIABLE (d de Cohen)")
print("=" * 78)
display(separacion[["variable", f"media_{clase_a}", f"media_{clase_b}",
                    "d_cohen", "magnitud", "efecto", "inclina_hacia"]])

orden = separacion["variable"].tolist()

Y ahora el gráfico, con las variables ordenadas según esa tabla: arriba a la izquierda la que
mejor separa, abajo a la derecha la que menos.

In [ ]:
n = len(orden)
filas = int(np.ceil(n / 3))
fig, ejes = plt.subplots(filas, 3, figsize=(14, 3.3 * filas))
ejes = np.array(ejes).ravel()

for i, col in enumerate(orden):
    for clase in clases:
        sns.kdeplot(df.loc[df[objetivo] == clase, col], ax=ejes[i],
                    fill=True, alpha=0.35, linewidth=1.8,
                    color=COLOR_CLASE.get(clase, None), label=str(clase))
    d = separacion.loc[separacion["variable"] == col, "d_cohen"].iloc[0]
    efecto = separacion.loc[separacion["variable"] == col, "efecto"].iloc[0]
    ejes[i].set_title(f"{col}\nd = {d:+.2f} ({efecto})", fontsize=10)
    ejes[i].set_xlabel("")
    ejes[i].set_ylabel("")
    ejes[i].legend(fontsize=8, title=objetivo)

for j in range(n, len(ejes)):
    ejes[j].axis("off")

fig.suptitle("Densidad por clase — ordenadas de mayor a menor separación",
             fontsize=13, fontweight="bold", y=1.00)
plt.tight_layout()
plt.show()

El mismo ordenamiento, visto como barras. Se lee de un golpe cuáles son las variables que van a
hacer el trabajo pesado en la Parte 2 y cuáles están casi mudas.

In [ ]:
fig, eje = plt.subplots(figsize=(10, 4.2))
colores_barras = [COLOR_CLASE.get(c, "#888888") for c in separacion["inclina_hacia"]]
eje.barh(separacion["variable"], separacion["d_cohen"], color=colores_barras,
         edgecolor="black", linewidth=0.6)
eje.axvline(0, color="black", linewidth=1)
for x in (-0.8, -0.5, -0.2, 0.2, 0.5, 0.8):
    eje.axvline(x, color="grey", linestyle=":", linewidth=0.8)
eje.set_xlabel(f"d de Cohen   (positivo inclina a {clase_a}, negativo a {clase_b})")
eje.set_title("Poder de separación de cada variable")
eje.invert_yaxis()
plt.tight_layout()
plt.show()

print("Las líneas punteadas marcan los umbrales convencionales de efecto chico (0,2),")
print("mediano (0,5) y grande (0,8).")

<a id="sec-3-7"></a>
## 3.7 Relaciones entre variables

Hasta acá miramos una variable por vez. Ahora las cruzamos de a dos, con un `pairplot`: una matriz
donde cada celda es un gráfico de dispersión entre dos variables, coloreado por clase, y la
diagonal muestra la distribución de cada una.

Qué buscar:

- **Nubes de puntos separadas por color** — ahí hay un par de variables que combinadas separan las
  clases mejor de lo que lo hace cada una sola.
- **Puntos alineados sobre una recta** — las dos variables dicen prácticamente lo mismo.
- **Puntos muy alejados del grupo** — outliers, pero vistos en dos dimensiones a la vez. Un caso
  puede tener valores perfectamente normales en cada variable por separado y ser muy raro en la
  combinación de las dos.

Con siete variables el gráfico completo tendría 49 celdas y sería ilegible. Usamos las cuatro que
mejor separan, según la tabla de 2.6.

In [ ]:
top4 = orden[:4]
print(f"Variables incluidas en el pairplot: {top4}")

grafico = sns.pairplot(df[top4 + [objetivo]], hue=objetivo, palette=COLOR_CLASE,
                       diag_kind="kde", height=2.0,
                       plot_kws={"s": 22, "alpha": 0.65,
                                 "edgecolor": "white", "linewidth": 0.3})
grafico.figure.suptitle("Relaciones entre las cuatro variables que mejor separan",
                        y=1.02, fontsize=13, fontweight="bold")
plt.show()

<a id="sec-3-8"></a>
## 3.8 Mapa de calor de correlaciones

La **correlación de Pearson** mide cuán fuerte es la relación lineal entre dos variables. Va de -1
a +1:

- **+1** — cuando una sube, la otra sube exactamente en la misma proporción.
- **0** — no hay relación lineal.
- **-1** — cuando una sube, la otra baja.

Por qué importa: si dos variables están muy correlacionadas, aportan información redundante. Eso
no arruina las predicciones, pero sí complica la **interpretación** de modelos lineales, porque el
peso que le corresponde a cada una se vuelve inestable —el modelo puede repartirlo entre las dos
de infinitas formas casi equivalentes—.

El mapa de calor muestra toda la matriz de un vistazo: rojo para correlación positiva, azul para
negativa, blanco para ausencia de relación.

In [ ]:
correlaciones = df[predictoras].corr()

fig, eje = plt.subplots(figsize=(8.5, 6.8))
mascara = np.triu(np.ones_like(correlaciones, dtype=bool), k=1)   # oculta la mitad repetida
sns.heatmap(correlaciones, mask=mascara, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5,
            cbar_kws={"shrink": 0.75, "label": "correlación de Pearson"}, ax=eje)
eje.set_title("Matriz de correlaciones entre variables predictoras")
plt.tight_layout()
plt.show()

pares_altos = []
for a, b in combinations(predictoras, 2):
    r = correlaciones.loc[a, b]
    if abs(r) >= UMBRAL_CORR_ALTA:
        pares_altos.append((a, b, r))

print("=" * 70)
print(f"  PARES CON |correlación| >= {UMBRAL_CORR_ALTA}")
print("=" * 70)
if pares_altos:
    for a, b, r in sorted(pares_altos, key=lambda t: -abs(t[2])):
        print(f"    {a:<12} <-> {b:<12}  r = {r:+.3f}")
    print()
    print("  Si quisiéramos simplificar el modelo, estos serían los candidatos a")
    print("  quedarse con una sola de las dos variables de cada par.")
else:
    print("    Ninguno. No hay redundancia grave entre las variables predictoras.")

print()
sin_diagonal = correlaciones.where(~np.eye(len(correlaciones), dtype=bool))
donde = sin_diagonal.abs().stack().idxmax()
print("  Correlación más fuerte encontrada, en valor absoluto:")
print(f"    {donde[0]} <-> {donde[1]}  r = {correlaciones.loc[donde[0], donde[1]]:+.3f}")

Un detalle de este dataset en particular, que explica el patrón que se ve en el mapa de calor:
**las siete columnas suman 1 en cada fila**. Son proporciones que reparten un total fijo.

Cuando un total está repartido, las variables no pueden moverse libremente: si una sube, alguna
otra necesariamente baja. Por eso la mayoría de las correlaciones son **negativas**, y no porque
haya una relación de causa y efecto entre ellas. Es una consecuencia aritmética de que el total
esté fijo.

Lo verificamos numéricamente en el punto 3.4.

<a id="sec-3-9"></a>
## 3.9 Un gráfico interactivo con Plotly

Todo lo anterior son gráficos **estáticos**: una imagen fija. Plotly produce gráficos
**interactivos**, que se pueden explorar con el mouse: hacer zoom, desplazarse, apagar y prender
series haciendo clic en la leyenda y —lo más útil— ver los valores exactos de cada punto al pasar
el cursor por encima.

La ventaja práctica es poder identificar casos concretos. En un gráfico estático vemos que hay un
punto raro; en uno interactivo le pasamos el cursor y sabemos exactamente qué fila es y qué valores
tiene.

Incluimos uno solo, a modo de ejemplo: las dos variables que mejor separan, una contra la otra,
coloreadas por clase.

Importamos Plotly acá, justo antes de usarlo, porque es la única sección que lo necesita.

In [ ]:
try:
    import plotly.express as px
    print(f"Plotly: {px.__name__} disponible")

    datos_plotly = df.copy()
    datos_plotly["fila"] = datos_plotly.index

    figura = px.scatter(
        datos_plotly,
        x=orden[0], y=orden[1],
        color=objetivo,
        color_discrete_map=COLOR_CLASE,
        hover_data=["fila"] + predictoras,
        opacity=0.78,
        title=f"Interactivo: {orden[0]} vs {orden[1]} (pasá el cursor sobre los puntos)",
        labels={objetivo: "Clase"},
    )
    figura.update_traces(marker=dict(size=10, line=dict(width=0.8, color="white")))
    figura.update_layout(width=880, height=520, legend_title_text="Clase")
    figura.show()

except ImportError:
    print("Plotly no está instalado en este entorno.")
    print("En Google Colab viene preinstalado. Fuera de Colab: pip install plotly")
    print("Se dibuja la versión estática equivalente:")
    fig, eje = plt.subplots(figsize=(8, 5.5))
    for clase in clases:
        sub = df[df[objetivo] == clase]
        eje.scatter(sub[orden[0]], sub[orden[1]], label=str(clase), alpha=0.75, s=45,
                    color=COLOR_CLASE.get(clase), edgecolor="white", linewidth=0.5)
    eje.set_xlabel(orden[0]); eje.set_ylabel(orden[1])
    eje.set_title(f"{orden[0]} vs {orden[1]}")
    eje.legend(title="Clase")
    plt.tight_layout(); plt.show()

---
# 4. Limpieza y consolidación

Terminado el EDA, viene la **auditoría de calidad**. Acá no exploramos: buscamos problemas
concretos, uno por uno, con una lista de control.

La lógica de esta sección conviene explicitarla: **vamos a buscar todos los defectos habituales aunque sepamos que este dataset no los tiene**. Encontrar cero duplicados no es lo mismo que
no haberlos buscado. Lo primero es un resultado; lo segundo es un agujero en el análisis que
después aparece como un problema inexplicable en el modelo.

Y en cada control vamos a dejar escrito, en código comentado, **cómo se resolvería el problema si
existiera**. Así el cuaderno sirve como referencia para el próximo dataset, que seguramente no va
a venir tan prolijo.

La lista de control:

| Punto | Control | Pregunta |
|---|---|---|
| 4.1 | Valores faltantes | ¿Hay celdas vacías? |
| 4.2 | Duplicados | ¿Hay filas repetidas? |
| 4.3 | Tipos de dato | ¿Cada columna es del tipo que corresponde? |
| 4.4 | Rangos y consistencia | ¿Los valores son posibles? ¿Hay columnas constantes? |
| 4.5 | Outliers | ¿Hay valores extremos? ¿Qué hacemos con ellos? |

<a id="sec-4-1"></a>
## 4.1 Valores faltantes

Un valor faltante (`NaN`, nulo) es una celda sin dato. Aparecen por mil razones: un formulario
incompleto, un sensor que se cayó, un campo que no aplicaba a ese registro, un error al unir dos
tablas.

Son un problema serio porque **la mayoría de los modelos de scikit-learn no los aceptan**: tiran
error directamente. Hay que resolverlos antes de modelar, sí o sí.

Lo verificamos de tres maneras: el conteo por columna, el conteo por fila, y un mapa visual donde
cada celda faltante aparecería como una marca roja.

In [ ]:
nulos_por_columna = df.isna().sum()
nulos_por_fila = df.isna().sum(axis=1)
total_nulos = int(nulos_por_columna.sum())

print("=" * 70)
print("  VALORES FALTANTES POR COLUMNA")
print("=" * 70)
tabla_nulos = pd.DataFrame({
    "nulos": nulos_por_columna,
    "porcentaje": (nulos_por_columna / len(df) * 100).round(2),
})
display(tabla_nulos)

print(f"  Total de celdas vacías                  : {total_nulos}")
print(f"  Filas con al menos un valor faltante    : {(nulos_por_fila > 0).sum()}")
print(f"  Columnas con al menos un valor faltante : {(nulos_por_columna > 0).sum()}")
print()
if total_nulos == 0:
    print("  RESULTADO: no hay valores faltantes. No hay nada que imputar ni que eliminar.")
else:
    print("  RESULTADO: hay valores faltantes. Ver la celda siguiente para el tratamiento.")

fig, eje = plt.subplots(figsize=(9, 4))
sns.heatmap(df.isna(), cbar=False, cmap=["#e8f4ea", "#d62728"], yticklabels=False, ax=eje)
eje.set_title("Mapa de valores faltantes  (rojo = faltante)")
plt.tight_layout()
plt.show()

**Si hubiera valores faltantes, así se tratarían.**

No hay una única receta correcta: la decisión depende de cuántos son, dónde están y qué
significan.

| Situación | Qué conviene hacer |
|---|---|
| Pocas filas afectadas (menos del 5%) y el dataset es grande | Eliminar esas filas: `dropna()` |
| Una columna con la gran mayoría de los valores vacíos | Eliminar la columna entera: es más ruido que información |
| Variable numérica, distribución simétrica | Imputar con la **media** |
| Variable numérica, distribución asimétrica o con outliers | Imputar con la **mediana** (es más robusta: no la arrastran los extremos) |
| Variable categórica | Imputar con la **moda**, o crear una categoría explícita `"desconocido"` |
| El hecho de que falte significa algo | Crear una columna indicadora `falta_X` con 0/1 antes de imputar, para no perder esa información |
| La etiqueta es la que falta | Eliminar la fila: sin etiqueta no sirve para entrenar |

**Una advertencia que se pasa por alto seguido.** Si vas a imputar, el valor con el que imputás
—la media, la mediana— **tiene que calcularse sobre el conjunto de entrenamiento, no sobre todo el
dataset**. Calcular la media usando también las filas de prueba es filtrar información del futuro
hacia el pasado. Eso se llama *fuga de información* y tiene una sección entera en la Parte 2.

Es una de las razones por las que este cuaderno **no imputa nada**: la imputación, como el
escalado, es una decisión del modelo y va adentro del pipeline, no en la limpieza.

El código de abajo está comentado porque en este dataset no hace falta.

In [ ]:
# ---------------------------------------------------------------------
# TRATAMIENTO DE VALORES FALTANTES  (comentado: acá no hace falta)
# ---------------------------------------------------------------------

# --- Opción 1: eliminar las filas que tengan algún faltante ---
# antes = len(df)
# df = df.dropna()
# print(f"Filas eliminadas: {antes - len(df)}")

# --- Opción 2: eliminar solo las filas sin etiqueta ---
# df = df.dropna(subset=[objetivo])

# --- Opción 3: eliminar columnas con demasiados faltantes ---
# UMBRAL = 0.40
# columnas_vacias = df.columns[df.isna().mean() > UMBRAL]
# df = df.drop(columns=columnas_vacias)

# --- Opción 4: imputar numéricas con la mediana ---
# for c in predictoras:
#     df[c] = df[c].fillna(df[c].median())

# --- Opción 5: imputar categóricas con la moda ---
# for c in df.select_dtypes(include="object").columns:
#     df[c] = df[c].fillna(df[c].mode()[0])

# --- Opción 6: dejar constancia de que faltaba, y después imputar ---
# for c in predictoras:
#     if df[c].isna().any():
#         df[f"falta_{c}"] = df[c].isna().astype(int)
#         df[c] = df[c].fillna(df[c].median())

# --- Opción 7 (la correcta para modelar, sin fuga: se ajusta solo con
#     el conjunto de entrenamiento; ver el punto 7.3) ---
# from sklearn.impute import SimpleImputer
# imputador = SimpleImputer(strategy="median")

print("Sin valores faltantes: no se aplica ningún tratamiento.")
print(f"El DataFrame sigue teniendo {df.shape[0]} filas y {df.shape[1]} columnas.")

<a id="sec-4-2"></a>
## 4.2 Registros duplicados

Una fila duplicada es una fila idéntica a otra. Suelen aparecer al unir tablas, al exportar dos
veces el mismo período, o por un error de carga.

Por qué molestan: un caso repetido **pesa el doble** en el entrenamiento, lo que sesga el modelo
hacia ese caso sin ninguna razón. Y hay algo peor: si el mismo caso queda en el conjunto de
entrenamiento y en el de prueba, el modelo va a "acertar" ese caso en la prueba porque lo
memorizó, no porque haya aprendido. La evaluación queda inflada y nadie se entera. (Esto también
es fuga de información, y vuelve a aparecer en la Parte 2.)

Hay que distinguir dos tipos de duplicado:

- **Duplicado completo** — la fila entera es idéntica. Casi siempre es un error.
- **Duplicado en las predictoras pero con etiqueta distinta** — dos casos con exactamente los
  mismos valores de entrada pero clasificados diferente. Esto es peor, porque es una
  **contradicción**: ningún modelo puede aprender a distinguirlos, y marcan un techo a la precisión
  alcanzable.

In [ ]:
duplicados_completos = df.duplicated()
duplicados_predictoras = df.duplicated(subset=predictoras)

print("=" * 70)
print("  REGISTROS DUPLICADOS")
print("=" * 70)
print(f"  Filas duplicadas completas (todas las columnas) : {int(duplicados_completos.sum())}")
print(f"  Filas duplicadas en las predictoras             : {int(duplicados_predictoras.sum())}")

if duplicados_predictoras.sum() > 0:
    grupos = df.groupby(predictoras, dropna=False)[objetivo].nunique()
    contradicciones = int((grupos > 1).sum())
else:
    contradicciones = 0
print(f"  Grupos con misma entrada y etiqueta distinta    : {contradicciones}")

print()
if duplicados_completos.sum() == 0 and contradicciones == 0:
    print("  RESULTADO: no hay duplicados ni contradicciones. No hay nada que eliminar.")
else:
    print("  RESULTADO: hay duplicados. Ver la celda siguiente.")
    display(df[df.duplicated(keep=False)].sort_values(predictoras))

**Si hubiera duplicados, así se eliminarían.**

`drop_duplicates()` conserva la primera aparición de cada fila repetida y descarta el resto. Con
`keep="last"` conserva la última, y con `keep=False` elimina todas las apariciones, incluida la
primera —útil cuando no se sabe cuál de las copias es la buena y se prefiere descartar el caso
entero—.

Para las contradicciones no hay solución automática: hay que ir al origen del dato y averiguar
cuál etiqueta es la correcta. Si eso no es posible, lo más honesto es eliminar esos casos y
dejarlo documentado.

In [ ]:
# ---------------------------------------------------------------------
# ELIMINACIÓN DE DUPLICADOS  (comentado: acá no hace falta)
# ---------------------------------------------------------------------

# --- Duplicados completos, conservando la primera aparición ---
# antes = len(df)
# df = df.drop_duplicates(keep="first").reset_index(drop=True)
# print(f"Filas eliminadas: {antes - len(df)}")

# --- Duplicados considerando solo las predictoras ---
# df = df.drop_duplicates(subset=predictoras, keep="first").reset_index(drop=True)

# --- Eliminar TODAS las apariciones de una fila duplicada ---
# df = df.drop_duplicates(keep=False).reset_index(drop=True)

# --- Contradicciones: mismas predictoras, etiquetas distintas ---
#     No se resuelven solas. Una salida es quedarse con la etiqueta mayoritaria
#     de cada grupo; otra, eliminar el grupo entero.
# conflictivos = df.groupby(predictoras)[objetivo].transform("nunique") > 1
# df = df[~conflictivos].reset_index(drop=True)

print("Sin duplicados: no se elimina ninguna fila.")
print(f"El DataFrame sigue teniendo {df.shape[0]} filas.")

<a id="sec-4-3"></a>
## 4.3 Tipos de dato

Una columna numérica leída como texto es el error silencioso más común de todos. Pasa cuando en
alguna celda hay un `"-"`, un `"N/D"`, un espacio, un símbolo de moneda o una coma decimal en un
archivo que espera punto. Con un solo valor así, Pandas declara la columna entera como `object` y
a partir de ahí todo lo que se calcule sobre ella está mal o directamente falla.

Verificamos que cada columna sea del tipo que le corresponde: la objetivo categórica, y las siete
predictoras numéricas.

In [ ]:
print("=" * 70)
print("  VERIFICACIÓN DE TIPOS")
print("=" * 70)

problemas_tipo = []

tipo_obj = df[objetivo].dtype
n_clases = df[objetivo].nunique()
print(f"  {objetivo:<12} tipo={str(tipo_obj):<10} valores únicos={n_clases}  -> ", end="")
if n_clases == 2:
    print("OK, categórica binaria")
elif n_clases < 15:
    print(f"categórica de {n_clases} clases (multiclase)")
else:
    print("CUIDADO: demasiados valores únicos para ser una etiqueta")
    problemas_tipo.append(objetivo)

for c in predictoras:
    es_num = pd.api.types.is_numeric_dtype(df[c])
    estado = "OK, numérica" if es_num else "PROBLEMA: no es numérica"
    if not es_num:
        problemas_tipo.append(c)
    print(f"  {c:<12} tipo={str(df[c].dtype):<10} valores únicos={df[c].nunique():<5} -> {estado}")

print()
if not problemas_tipo:
    print("  RESULTADO: todos los tipos son los correctos. No hay conversiones que hacer.")
else:
    print(f"  RESULTADO: revisar estas columnas -> {problemas_tipo}")

**Si alguna columna tuviera el tipo equivocado, así se corregiría.**

La clave es `pd.to_numeric(..., errors="coerce")`: intenta convertir a número y lo que no puede
convertir lo transforma en `NaN`. Después se mira cuántos `NaN` aparecieron y, si son pocos, se
tratan como valores faltantes con lo visto en 3.1. Si son muchos, conviene revisar el archivo de
origen antes de seguir.

In [ ]:
# ---------------------------------------------------------------------
# CORRECCIÓN DE TIPOS  (comentado: acá no hace falta)
# ---------------------------------------------------------------------

# --- Texto a número; lo que no se puede convertir queda como NaN ---
# df["columna"] = pd.to_numeric(df["columna"], errors="coerce")

# --- Limpiar símbolos y separadores antes de convertir ---
# df["precio"] = (df["precio"].astype(str)
#                              .str.replace("$", "", regex=False)
#                              .str.replace(".", "", regex=False)   # separador de miles
#                              .str.replace(",", ".", regex=False)  # decimal
#                              .str.strip())
# df["precio"] = pd.to_numeric(df["precio"], errors="coerce")

# --- Normalizar una columna de texto: espacios y mayúsculas ---
# df[objetivo] = df[objetivo].astype(str).str.strip().str.upper()

# --- Texto a fecha ---
# df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce", dayfirst=True)

print("Tipos correctos: no se aplica ninguna conversión.")
print()
print("Chequeo extra: espacios o mayúsculas inconsistentes en la variable objetivo")
valores_crudos = df[objetivo].astype(str)
valores_limpios = valores_crudos.str.strip().str.upper()
diferencias = int((valores_crudos != valores_limpios).sum())
print(f"  Valores que cambiarían al limpiarlos : {diferencias}")
print(f"  Valores únicos tal como vienen       : {sorted(valores_crudos.unique())}")

<a id="sec-4-4"></a>
## 4.4 Rangos y consistencia

Este control es el que más depende de conocer los datos. No se trata de buscar errores genéricos
sino de preguntarse: **¿estos valores son posibles?**

Cuatro verificaciones:

1. **Columnas constantes o casi constantes.** Una columna con un solo valor no aporta absolutamente
   nada a ningún modelo, y una con casi un solo valor aporta muy poco. Se descartan.
2. **Valores fuera de rango.** En este dataset las siete columnas son proporciones, así que
   deberían estar entre 0 y 1. Un negativo o un valor mayor a 1 sería un error.
3. **Coherencia entre columnas.** Si las columnas reparten un total, ese total tiene que dar. Acá
   verificamos que las siete sumen 1 en cada fila.
4. **Cardinalidad del objetivo.** Que las clases sean las que esperamos, sin variantes escritas de
   distinta forma.

In [ ]:
print("=" * 78)
print("  1. COLUMNAS CONSTANTES O CASI CONSTANTES")
print("=" * 78)
constantes = [c for c in predictoras if df[c].nunique() <= 1]
casi_constantes = [c for c in predictoras
                   if df[c].nunique() > 1 and df[c].value_counts(normalize=True).iloc[0] > 0.95]
print(f"  Constantes      : {constantes if constantes else 'ninguna'}")
print(f"  Casi constantes : {casi_constantes if casi_constantes else 'ninguna'}")
print(f"  Desvío mínimo entre las predictoras: {df[predictoras].std().min():.6f}")

print()
print("=" * 78)
print("  2. VALORES FUERA DE RANGO")
print("=" * 78)
negativos = int((df[predictoras] < 0).sum().sum())
mayores_a_uno = int((df[predictoras] > 1).sum().sum())
print(f"  Valores negativos     : {negativos}")
print(f"  Valores mayores que 1 : {mayores_a_uno}")
print(f"  Mínimo global         : {df[predictoras].to_numpy().min():.6f}")
print(f"  Máximo global         : {df[predictoras].to_numpy().max():.6f}")

print()
print("=" * 78)
print("  3. COHERENCIA: ¿SUMAN 1 LAS FILAS?")
print("=" * 78)
sumas = df[predictoras].sum(axis=1)
tolerancia = 1e-6
fuera = int((~np.isclose(sumas, 1.0, atol=tolerancia)).sum())
print(f"  Suma mínima por fila : {sumas.min():.10f}")
print(f"  Suma máxima por fila : {sumas.max():.10f}")
print(f"  Desvío de las sumas  : {sumas.std():.2e}")
print(f"  Filas que NO suman 1 (tolerancia {tolerancia:g}) : {fuera}")
if fuera == 0:
    print()
    print("  Confirmado: las siete columnas reparten un total fijo de 1.")
    print("  Esto explica las correlaciones negativas que vimos en 3.8: si una")
    print("  proporción sube, alguna otra tiene que bajar. Es aritmética, no una")
    print("  relación de causa y efecto.")

print()
print("=" * 78)
print("  4. CARDINALIDAD DE LA VARIABLE OBJETIVO")
print("=" * 78)
print(f"  Clases encontradas : {sorted(df[objetivo].unique())}")
print(f"  Cantidad de clases : {df[objetivo].nunique()}")

**Si algo de lo anterior fallara, así se corregiría.**

In [ ]:
# ---------------------------------------------------------------------
# CORRECCIÓN DE RANGOS Y CONSISTENCIA  (comentado: acá no hace falta)
# ---------------------------------------------------------------------

# --- Eliminar columnas constantes ---
# constantes = [c for c in df.columns if df[c].nunique() <= 1]
# df = df.drop(columns=constantes)
# predictoras = [c for c in predictoras if c not in constantes]

# --- Filas con valores imposibles: eliminarlas ---
# invalidas = (df[predictoras] < 0).any(axis=1)
# print(f"Filas con valores negativos: {invalidas.sum()}")
# df = df[~invalidas].reset_index(drop=True)

# --- Recortar valores al rango válido en lugar de eliminar la fila ---
# df[predictoras] = df[predictoras].clip(lower=0, upper=1)

# --- Si las proporciones no sumaran 1, renormalizarlas ---
# df[predictoras] = df[predictoras].div(df[predictoras].sum(axis=1), axis=0)

# --- Unificar etiquetas escritas de distinta forma ---
# df[objetivo] = df[objetivo].replace({"afi": "AFI", "Afi": "AFI", "neg": "NEG"})

print("Rangos y consistencia correctos: no se aplica ninguna corrección.")

<a id="sec-4-5"></a>
## 4.5 Outliers: detección y decisión

Un **outlier** (valor atípico) es un dato que se aleja mucho del comportamiento general del resto.
Puede tener tres orígenes muy distintos, y de eso depende qué hacer con él:

1. **Un error de carga.** Alguien escribió 25000 donde iba 250. Hay que corregirlo o eliminarlo.
2. **Un caso real pero excepcional.** Una venta enorme que efectivamente ocurrió. Puede convenir
   conservarla para que el modelo aprenda a manejar esos casos.
3. **Un caso real e informativo.** El valor extremo es precisamente lo que distingue a ese caso.
   Eliminarlo sería tirar la señal.

Vamos a usar dos métodos de detección:

- **Regla del rango intercuartílico (IQR), o regla de Tukey.** Se considera atípico todo valor
  fuera del intervalo `[Q1 - 1,5 x IQR , Q3 + 1,5 x IQR]`. Es la regla que dibujan los boxplots.
  No supone ninguna forma de distribución y no se deja arrastrar por los propios extremos.
- **Puntaje z (z-score).** Se considera atípico todo valor a más de 3 desvíos estándar de la media.
  Es más simple de explicar, pero tiene un problema: la media y el desvío se calculan *con* los
  outliers adentro, así que los propios extremos inflan el umbral y ayudan a esconderse.

In [ ]:
def outliers_iqr(serie, factor=FACTOR_IQR):
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    iqr = q3 - q1
    bajo, alto = q1 - factor * iqr, q3 + factor * iqr
    return (serie < bajo) | (serie > alto), bajo, alto


def outliers_z(serie, umbral=UMBRAL_Z):
    media, desvio = serie.mean(), serie.std()
    if desvio == 0:
        return pd.Series(False, index=serie.index)
    return ((serie - media).abs() / desvio) > umbral


resumen_outliers = []
mascara_iqr = pd.DataFrame(False, index=df.index, columns=predictoras)
mascara_z = pd.DataFrame(False, index=df.index, columns=predictoras)

for c in predictoras:
    m_iqr, bajo, alto = outliers_iqr(df[c])
    m_z = outliers_z(df[c])
    mascara_iqr[c] = m_iqr
    mascara_z[c] = m_z
    resumen_outliers.append({
        "variable": c,
        "limite_inferior": bajo,
        "limite_superior": alto,
        "outliers_IQR": int(m_iqr.sum()),
        "pct_IQR": m_iqr.mean() * 100,
        "outliers_z": int(m_z.sum()),
        "pct_z": m_z.mean() * 100,
    })

resumen_outliers = pd.DataFrame(resumen_outliers).sort_values("outliers_IQR", ascending=False)

print("=" * 78)
print("  OUTLIERS POR VARIABLE")
print("=" * 78)
display(resumen_outliers)

filas_con_outlier = mascara_iqr.any(axis=1)
print(f"  Filas con al menos un outlier (IQR) : {int(filas_con_outlier.sum())} "
      f"de {len(df)}  ({filas_con_outlier.mean():.1%})")
print(f"  Filas con al menos un outlier (z)   : {int(mascara_z.any(axis=1).sum())} "
      f"de {len(df)}  ({mascara_z.any(axis=1).mean():.1%})")
print(f"  Total de celdas marcadas (IQR)      : {int(mascara_iqr.to_numpy().sum())} "
      f"de {df[predictoras].size}")

Verlos es más útil que contarlos. Dibujamos cada variable con sus puntos marcados: en gris los
valores normales, en rojo los que la regla IQR señala como atípicos.

In [ ]:
n = len(predictoras)
filas = int(np.ceil(n / 3))
fig, ejes = plt.subplots(filas, 3, figsize=(14, 3.2 * filas))
ejes = np.array(ejes).ravel()

for i, col in enumerate(predictoras):
    normales = df.loc[~mascara_iqr[col], col]
    atipicos = df.loc[mascara_iqr[col], col]
    ejes[i].scatter(normales.index, normales.values, s=16, alpha=0.55,
                    color="#888888", label="normal")
    if len(atipicos):
        ejes[i].scatter(atipicos.index, atipicos.values, s=42, alpha=0.95,
                        color="#d62728", edgecolor="black", linewidth=0.5,
                        label=f"atípico ({len(atipicos)})")
    fila = resumen_outliers[resumen_outliers["variable"] == col].iloc[0]
    ejes[i].axhline(fila["limite_superior"], color="#d62728", linestyle="--", linewidth=1)
    ejes[i].axhline(fila["limite_inferior"], color="#d62728", linestyle="--", linewidth=1)
    ejes[i].set_title(col)
    ejes[i].set_xlabel("índice de la fila")
    ejes[i].legend(fontsize=7)

for j in range(n, len(ejes)):
    ejes[j].axis("off")

fig.suptitle("Outliers según la regla IQR  (líneas punteadas = límites de Tukey)",
             fontsize=13, fontweight="bold", y=1.00)
plt.tight_layout()
plt.show()

Y una mirada más, que es la que decide el asunto: **¿los outliers se reparten parejo entre las dos
clases?**

Si los valores extremos aparecieran sobre todo en una clase, eliminarlos significaría borrar
justamente los casos más característicos de esa clase —y el modelo empeoraría por culpa de la
"limpieza"—.

In [ ]:
outliers_por_clase = pd.DataFrame({
    "casos": df[objetivo].value_counts(),
    "con_outlier": df.loc[filas_con_outlier, objetivo].value_counts(),
})
outliers_por_clase["con_outlier"] = outliers_por_clase["con_outlier"].fillna(0).astype(int)
outliers_por_clase["porcentaje"] = (outliers_por_clase["con_outlier"] /
                                    outliers_por_clase["casos"] * 100).round(1)

print("=" * 70)
print("  REPARTO DE LOS OUTLIERS ENTRE LAS CLASES")
print("=" * 70)
display(outliers_por_clase)

fig, ejes = plt.subplots(1, 2, figsize=(13, 4.2))

ejes[0].bar(outliers_por_clase.index.astype(str), outliers_por_clase["porcentaje"],
            color=[COLOR_CLASE.get(c, "#888") for c in outliers_por_clase.index],
            edgecolor="black", linewidth=0.6)
ejes[0].set_title("% de casos con al menos un outlier, por clase")
ejes[0].set_ylabel("%")

cantidad_outliers = mascara_iqr.sum(axis=1)
for clase in clases:
    sns.histplot(cantidad_outliers[df[objetivo] == clase], bins=range(0, 6),
                 ax=ejes[1], alpha=0.55, label=str(clase),
                 color=COLOR_CLASE.get(clase), discrete=True)
ejes[1].set_title("Cantidad de variables atípicas por caso")
ejes[1].set_xlabel("variables marcadas como atípicas en ese caso")
ejes[1].legend(title=objetivo)

plt.tight_layout()
plt.show()

**La decisión: los detectamos, los mostramos y no los eliminamos.**

Los motivos, en orden:

1. **No son errores de carga.** La auditoría de 4.4 mostró que todos los valores están dentro del
   rango posible (entre 0 y 1) y que cada fila suma exactamente 1. Un error de carga rompería
   alguna de esas dos condiciones. Estos valores son perfectamente válidos.

2. **Acá un "outlier" es un caso concentrado, no un caso raro.** Como las siete columnas reparten
   un total fijo, que una de ellas tenga un valor alto significa que ese caso está concentrado en
   esa variable. Eso no es una anomalía: es exactamente el tipo de información que permite
   distinguir una clase de otra. Eliminarlos sería tirar la señal y quedarse con el ruido.

3. **El dataset es chico.** Con 171 filas, cada caso descartado es casi el 0,6% de la muestra. La
   regla IQR marca una proporción de filas que no se puede permitir perder sin una razón mucho más
   fuerte que "el boxplot los dibujó afuera de los bigotes".

4. **Los modelos de la Parte 2 los toleran.** La regresión logística con regularización, los
   árboles y los bosques no se desestabilizan por unos pocos valores extremos. KNN sí es más
   sensible, y eso va a quedar registrado en la comparación final: es un resultado del experimento,
   no algo que haya que arreglar de antemano.

*Lo más importante es no aplicar soluciones automáticas*. Un
outlier no es un defecto por definición. Es un dato que hay que mirar.

Igual dejamos escrito, comentado, cómo se trataría si la decisión hubiera sido otra.

In [ ]:
# ---------------------------------------------------------------------
# TRATAMIENTO DE OUTLIERS  (comentado a propósito: NO los eliminamos)
# ---------------------------------------------------------------------

# --- Opción 1: eliminar las filas con algún outlier ---
# antes = len(df)
# df = df[~filas_con_outlier].reset_index(drop=True)
# print(f"Filas eliminadas: {antes - len(df)}")

# --- Opción 2: eliminar solo los casos extremos en varias variables a la vez ---
# muy_atipicos = mascara_iqr.sum(axis=1) >= 3
# df = df[~muy_atipicos].reset_index(drop=True)

# --- Opción 3: "capar" (winsorizar): limitar el valor al límite en vez de borrar la fila ---
# for c in predictoras:
#     q1, q3 = df[c].quantile(0.25), df[c].quantile(0.75)
#     iqr = q3 - q1
#     df[c] = df[c].clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

# --- Opción 4: capar por percentiles (winsorización al 1%-99%) ---
# for c in predictoras:
#     bajo, alto = df[c].quantile(0.01), df[c].quantile(0.99)
#     df[c] = df[c].clip(bajo, alto)

# --- Opción 5: transformar para suavizar el efecto de los extremos ---
# for c in predictoras:
#     df[f"{c}_log"] = np.log1p(df[c])

# --- Opción 6: escalado robusto, que usa mediana e IQR (ver el punto 5.2) ---
# from sklearn.preprocessing import RobustScaler

print("DECISIÓN: se conservan todos los casos.")
print(f"El DataFrame mantiene sus {df.shape[0]} filas.")

<a id="sec-4-6"></a>
## 4.6 Resumen de la auditoría de calidad

La lista de control completa con su resultado. Esta tabla es el entregable de la etapa: deja
constancia de qué se revisó, no solo de qué se encontró.

In [ ]:
auditoria = pd.DataFrame([
    {"control": "4.1 Valores faltantes",
     "encontrado": f"{total_nulos} celdas vacías", "accion": "ninguna"},
    {"control": "4.2 Duplicados completos",
     "encontrado": f"{int(duplicados_completos.sum())} filas", "accion": "ninguna"},
    {"control": "4.2 Contradicciones (misma entrada, distinta etiqueta)",
     "encontrado": f"{contradicciones} grupos", "accion": "ninguna"},
    {"control": "4.3 Tipos de dato incorrectos",
     "encontrado": f"{len(problemas_tipo)} columnas", "accion": "ninguna"},
    {"control": "4.4 Columnas constantes",
     "encontrado": f"{len(constantes)} columnas", "accion": "ninguna"},
    {"control": "4.4 Valores fuera de rango",
     "encontrado": f"{negativos + mayores_a_uno} valores", "accion": "ninguna"},
    {"control": "4.4 Filas que no suman 1",
     "encontrado": f"{fuera} filas", "accion": "ninguna"},
    {"control": "4.5 Filas con algún outlier (IQR)",
     "encontrado": f"{int(filas_con_outlier.sum())} filas",
     "accion": "SE CONSERVAN (ver justificación en 4.5)"},
])

print("=" * 88)
print("  AUDITORÍA DE CALIDAD — RESULTADO")
print("=" * 88)
display(auditoria)

print(f"  El dataset sale de la auditoría con {df.shape[0]} filas y "
      f"{len(predictoras)} variables predictoras.")
print("  No se eliminó ni se modificó ningún dato.")

---
# 5. Transformaciones básicas

Los datos están explorados y auditados. Ahora hay que **prepararlos para el modelo**. Esta etapa
no se ve en el resultado final, pero decide buena parte de él: los algoritmos no trabajan con
datos crudos, necesitan información numérica, en escalas comparables y sin ambigüedades.

Tres tareas:

| Punto | Tarea | Qué resuelve |
|---|---|---|
| 5.1 | Codificación de variables categóricas | Los algoritmos no leen texto |
| 5.2 | Escalado de variables numéricas | Las variables con números grandes dominan a las chicas |
| 5.3 | Creación o eliminación de variables | ¿Conviene fabricar variables nuevas? |

**Una aclaración sobre el orden que conviene leer antes de seguir.**

Codificar y escalar no son la misma clase de operación, aunque las dos aparezcan juntas en todos
los manuales:

- **Codificar no aprende nada de los datos.** `AFI` pasa a 1 y `NEG` a 0, y punto. Es un reemplazo
  fijo, así que se puede hacer ahora, sobre todo el dataset, sin ninguna consecuencia.
- **Escalar sí aprende parámetros.** Min-Max aprende el mínimo y el máximo; Z-score aprende la
  media y el desvío. Si esos parámetros se calculan sobre el dataset completo, el conjunto de
  prueba influye sobre la transformación que se aplica al de entrenamiento. Eso se llama **fuga de
  información** (*data leakage*) y hace que la evaluación salga mejor de lo que realmente es.

Por eso en este punto el escalado se **explica y se muestra**, pero no se aplica. Se aplica en el
punto 7.3, después de dividir, ajustándolo solo con el conjunto de entrenamiento. Es una decisión
deliberada y vale la pena dejarla escrita.

<a id="sec-5-1"></a>
## 5.1 Codificación de variables categóricas

Una variable categórica toma un conjunto limitado de valores, normalmente texto. Pueden ser:

- **Nominales**: sin orden entre ellas (rojo, azul, verde).
- **Ordinales**: con una jerarquía (bajo, medio, alto).

Hay dos técnicas para convertirlas en números:

**Label Encoding** — asigna un entero a cada categoría: `AFI → 1`, `NEG → 0`. Es directo y no
agrega columnas. El riesgo: introduce un **orden** donde puede no haberlo. Si codificamos
`rojo=0, azul=1, verde=2`, el modelo puede interpretar que verde es "más" que rojo, lo cual es un
sinsentido. Por eso solo conviene en variables ordinales.

**One-Hot Encoding** — crea una columna binaria por categoría. No introduce ningún orden, así que
es la opción segura para nominales. El costo: con muchas categorías genera muchísimas columnas; en
ese caso conviene agrupar las poco frecuentes en un "Otros".

**Qué corresponde acá.** La única columna de texto del dataset es la **variable objetivo**, y para
el objetivo la regla es distinta que para las predictoras: en clasificación binaria se codifica
**siempre** con Label Encoding, a 0 y 1. Con dos clases no hay jerarquía falsa posible, y One-Hot
generaría dos columnas perfectamente redundantes —una sería `1 -` la otra—.

Las siete predictoras ya son numéricas: no hay nada que codificar ahí.

In [ ]:
# --- Herramienta de codificación ---
from sklearn.preprocessing import LabelEncoder

# Fijamos explícitamente qué clase es la positiva (1) en lugar de dejarlo al
# azar del orden alfabético. La elección es arbitraria, pero NO es indiferente:
# la precisión, el recall y el F1 se calculan respecto de la clase positiva.
clase_negativa = [c for c in clases if c != CLASE_POSITIVA][0]
mapa_clases = {clase_negativa: 0, CLASE_POSITIVA: 1}
mapa_inverso = {v: k for k, v in mapa_clases.items()}

y = df[objetivo].map(mapa_clases).astype(int)

print("=" * 72)
print("  CODIFICACIÓN DE LA VARIABLE OBJETIVO (Label Encoding)")
print("=" * 72)
for etiqueta, codigo in mapa_clases.items():
    rol = "POSITIVA" if codigo == 1 else "negativa"
    print(f"  {etiqueta:>5}  ->  {codigo}   (clase {rol})   {int((y == codigo).sum())} casos")
print()
print(f"  Todas las métricas orientadas a una clase se van a reportar")
print(f"  respecto de la clase positiva: {CLASE_POSITIVA}")

# El mismo resultado con LabelEncoder de scikit-learn, para conocer la herramienta.
# Ojo: LabelEncoder asigna los números por orden alfabético, sin preguntar.
codificador = LabelEncoder()
y_sklearn = codificador.fit_transform(df[objetivo])
print()
print(f"  Con LabelEncoder de scikit-learn el mapeo sería: "
      f"{dict(zip(codificador.classes_, range(len(codificador.classes_))))}")
print(f"  ¿Coincide con el nuestro?: {bool((y_sklearn == y.to_numpy()).all())}")

# --- One-Hot, solo para mostrar la diferencia ---
demo_onehot = pd.get_dummies(df[[objetivo]], columns=[objetivo], prefix="clase").astype(int)
print()
print("=" * 72)
print("  ONE-HOT ENCODING (demostración, NO se usa)")
print("=" * 72)
display(pd.concat([df[[objetivo]].head(5), demo_onehot.head(5)], axis=1))
print(f"  Una columna de texto se convirtió en {demo_onehot.shape[1]} columnas binarias,")
print("  y las dos son complementarias: una es 1 menos la otra. Para un objetivo")
print("  binario eso es información duplicada.")
print()
print("  La forma habitual de usarlo, sobre predictoras nominales, sería:")
print('      df = pd.get_dummies(df, columns=["color", "ciudad"], drop_first=True)')
print("  drop_first=True elimina una columna por variable para evitar la redundancia.")

<a id="sec-5-2"></a>
## 5.2 Escalado de variables numéricas

La **escala** de una variable es el rango de valores que puede tomar. Cuando unas variables van de
0 a 1 y otras de 0 a 100.000, algunos modelos le dan más peso a las segundas solo porque sus
números son más grandes, no porque sean más importantes.

Las dos técnicas principales:

**Normalización (Min-Max Scaling)**

$$X_{norm} = \frac{X - X_{min}}{X_{max} - X_{min}}$$

Lleva todos los valores al rango 0–1. Conserva la forma exacta de la distribución. Es la opción
natural cuando el rango está acotado y para modelos basados en distancias, como KNN. Su punto
débil: un solo valor extremo define el máximo y comprime todo lo demás contra el cero.

**Estandarización (Z-score Scaling)**

$$X_{std} = \frac{X - \mu}{\sigma}$$

Centra los datos en media 0 y desvío 1. La mayoría de los valores queda entre -2 y 2. No acota el
rango. Es la opción habitual cuando no hay un rango fijo y para modelos que suponen distribuciones
centradas, como la regresión logística.

**Y hay más, por si el dataset lo pide:**

| Escalador | Qué hace | Ventaja | Desventaja |
|---|---|---|---|
| `RobustScaler` | Resta la mediana y divide por el IQR | Los outliers casi no lo afectan | No acota el rango |
| `MaxAbsScaler` | Divide por el máximo valor absoluto | Preserva los ceros; sirve para datos ralos | Muy sensible a un único extremo |
| `PowerTransformer` | Yeo-Johnson o Box-Cox | Corrige asimetría | Deforma la escala; difícil de interpretar |
| `QuantileTransformer` | Fuerza una distribución uniforme o normal | Neutraliza outliers | Agresivo; necesita bastantes datos |
| `np.log1p` | Logaritmo de (1 + x) | Simple contra colas largas | Solo valores no negativos |

**Qué modelos lo necesitan:**

| Modelo | ¿Escalado? | Por qué |
|---|---|---|
| Regresión Logística | **Sí** | La regularización penaliza los coeficientes por igual |
| KNN | **Sí, imprescindible** | Todas sus decisiones son distancias |
| SVM | **Sí** | Trabaja con distancias y márgenes |
| Redes neuronales | **Sí** | El descenso de gradiente converge mucho mejor |
| Árbol de Decisión | **No** | Corta por umbrales; una transformación monótona no cambia el corte |
| Random Forest / Gradient Boosting | **No** | Son conjuntos de árboles |

Escalar no agrega información: cambia cómo el modelo la interpreta.

**Observación propia sobre este dataset.** Las siete variables ya están en el mismo rango —todas
son proporciones entre 0 y 1— así que el escalado tiene poco margen de mejora acá. No es motivo
para saltearlo: se aplica igual, porque cuesta nada y porque algunos modelos lo necesitan por
construcción. Pero conviene anticipar que el efecto va a ser chico, para no atribuirle después una
mejora que no produjo.

La celda de abajo **no modifica nada**: trabaja sobre una copia y solo muestra el efecto de cada
escalador. La aplicación real es el punto 7.3.

In [ ]:
# --- Los escaladores ---
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# OJO: esto es una DEMOSTRACIÓN sobre una copia. No modifica df.
ejemplo = df[predictoras].copy()

minmax_demo = pd.DataFrame(MinMaxScaler().fit_transform(ejemplo),
                           columns=predictoras, index=ejemplo.index)
zscore_demo = pd.DataFrame(StandardScaler().fit_transform(ejemplo),
                           columns=predictoras, index=ejemplo.index)
robusto_demo = pd.DataFrame(RobustScaler().fit_transform(ejemplo),
                            columns=predictoras, index=ejemplo.index)

comparacion = pd.DataFrame({
    "original_min": ejemplo.min(),
    "original_max": ejemplo.max(),
    "original_media": ejemplo.mean(),
    "original_desvio": ejemplo.std(),
    "minmax_min": minmax_demo.min(),
    "minmax_max": minmax_demo.max(),
    "zscore_media": zscore_demo.mean(),
    "zscore_desvio": zscore_demo.std(ddof=0),
})

print("=" * 88)
print("  EFECTO DE CADA ESCALADOR")
print("=" * 88)
display(comparacion)
print("  Min-Max deja todo entre 0 y 1.  Z-score deja media 0 y desvío 1.")
print("  Ninguno de los dos cambia la FORMA de la distribución: solo el eje.")

fig, ejes = plt.subplots(1, 4, figsize=(17, 4))
for eje, datos, titulo in zip(
        ejes,
        [ejemplo, minmax_demo, zscore_demo, robusto_demo],
        ["Original", "Min-Max (0 a 1)", "Z-score (media 0, desvío 1)", "Robusto (mediana e IQR)"]):
    datos_largos = datos.melt(var_name="variable", value_name="valor")
    sns.boxplot(data=datos_largos, x="variable", y="valor", ax=eje,
                color="#4c72b0", fliersize=2.5)
    eje.set_title(titulo, fontsize=10)
    eje.tick_params(axis="x", rotation=90)
    eje.set_xlabel("")
plt.tight_layout()
plt.show()

print(f"  df NO se modificó: sigue con {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"  y su máximo original de {df[predictoras].to_numpy().max():.4f}.")
print("  El escalado de verdad se aplica en el punto 7.3.")

<a id="sec-5-3"></a>
## 5.3 Creación o eliminación de variables

**Feature engineering** es crear, seleccionar, transformar o eliminar variables para que el modelo
tenga una mejor representación del problema. Los ejemplos clásicos: extraer el mes y el día de la
semana de una fecha, o calcular el margen restando el costo del precio de venta.

Se suele decir que una buena ingeniería de características mejora más el resultado que cambiar de
algoritmo. Es cierto **cuando hay información latente que el modelo no puede deducir solo**.

Acá no hay fechas ni campos compuestos, pero sí se pueden construir variables que resuman la
**forma del reparto** de cada fila. Probamos tres:

- **`maximo`** — el valor de la variable dominante en ese caso. Mide cuán concentrado está.
- **`brecha`** — la diferencia entre la variable más alta y la segunda. Mide cuán clara es esa
  dominancia.
- **`entropia`** — cuán repartido está el total entre las siete variables. Baja cuando casi todo
  está en una sola, alta cuando está parejo.

Las construimos, medimos su poder de separación con la misma vara del punto 3.6 —la **d de
Cohen**— y recién ahí decidimos.

In [ ]:
proporciones = df[predictoras].to_numpy()
ordenadas = np.sort(proporciones, axis=1)[:, ::-1]      # de mayor a menor

nuevas = pd.DataFrame(index=df.index)
nuevas["maximo"] = ordenadas[:, 0]
nuevas["brecha"] = ordenadas[:, 0] - ordenadas[:, 1]
seguro = np.clip(proporciones, 1e-12, None)
nuevas["entropia"] = -(seguro * np.log(seguro)).sum(axis=1)

evaluacion = pd.DataFrame({
    "variable": nuevas.columns,
    "d_cohen": [d_de_cohen(nuevas[c], df[objetivo], clase_a, clase_b) for c in nuevas.columns],
})
evaluacion["magnitud"] = evaluacion["d_cohen"].abs()
evaluacion["efecto"] = evaluacion["magnitud"].map(etiqueta_efecto)

print("=" * 72)
print("  VARIABLES DERIVADAS: ¿aportan separación?")
print("=" * 72)
display(evaluacion)
print(f"  Mejor variable ORIGINAL : |d| = {separacion['magnitud'].max():.3f}")
print(f"  Mejor variable DERIVADA : |d| = {evaluacion['magnitud'].max():.3f}")

fig, ejes = plt.subplots(1, 3, figsize=(14, 3.6))
for i, col in enumerate(nuevas.columns):
    for clase in clases:
        sns.kdeplot(nuevas.loc[df[objetivo] == clase, col], ax=ejes[i],
                    fill=True, alpha=0.35, linewidth=1.8,
                    color=COLOR_CLASE.get(clase), label=str(clase))
    d = evaluacion.loc[evaluacion["variable"] == col, "d_cohen"].iloc[0]
    ejes[i].set_title(f"{col}  (d = {d:+.2f})")
    ejes[i].set_xlabel(""); ejes[i].set_ylabel("")
    ejes[i].legend(fontsize=8, title=objetivo)
fig.suptitle("Las tres variables derivadas, por clase", fontsize=12, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

**Decisión: no las incorporamos.**

Tres razones:

1. **No traen información nueva.** Las tres se calculan enteramente a partir de las siete columnas
   originales. Un modelo no lineal —un árbol, un bosque— puede reconstruir esas relaciones solo si
   le sirven. Agregarlas es repetirle algo que ya tiene.
2. **Agregan redundancia.** Cada derivada está fuertemente correlacionada con las originales de
   las que sale, y eso desestabiliza los coeficientes de un modelo lineal sin mejorar sus
   predicciones.
3. **Ensucian la comparación.** En la Parte 2 se van a comparar varios modelos sobre las mismas
   entradas. Si además cambiara el conjunto de variables, no se podría saber a qué atribuir una
   diferencia.

Esto **no** es una regla general: en la mayoría de los problemas reales el feature engineering es
de lo que más rinde. Acá no aplica porque las siete variables ya son una descripción completa de
cada caso.

Si más adelante quisieras probarlas, alcanza con descomentar las dos líneas de abajo: el resto del
cuaderno las toma solo, porque todo trabaja sobre la lista `predictoras`.

In [ ]:
# --- Para incorporar las variables derivadas, descomentá estas dos líneas ---
# df = pd.concat([df, nuevas], axis=1)
# predictoras = predictoras + list(nuevas.columns)

print(f"Variables predictoras en uso ({len(predictoras)}):")
for c in predictoras:
    print(f"    - {c}")
print()
print("Variables eliminadas: ninguna. La auditoría del punto 4.4 no encontró")
print("columnas constantes ni identificadores, que son los dos casos en los que")
print("una columna se descarta sin discusión.")

---
# 6. Selección de variables relevantes

Seleccionar variables es quedarse con las que aportan y descartar las que solo agregan ruido.
Cuando funciona, da modelos más simples, más rápidos, más fáciles de explicar y —a veces— más
precisos, porque menos columnas significa menos oportunidades de que el modelo encuentre
casualidades.

<a id="sec-6-1"></a>
## 6.1 Tres criterios de relevancia

Ninguna medida sola alcanza, así que usamos tres que miran cosas distintas:

**1. d de Cohen** — la diferencia entre las medias de las dos clases, dividida por el desvío
combinado. Ya la calculamos en el punto 3.6. Mide **cuánto se corren** las distribuciones de una
clase respecto de la otra. Al dividir por el desvío no depende de la escala, así que permite
comparar variables distintas. Solo capta diferencias de **promedio**.

**2. ANOVA F (`f_classif`)** — el estadístico F de un análisis de varianza de un factor. Responde
la misma pregunta que la d de Cohen desde otro ángulo, pero además devuelve un **p-valor**: la
probabilidad de ver una diferencia así de grande si en realidad no hubiera ninguna. También es una
medida lineal, de diferencia de medias.

**3. Información mutua (`mutual_info_classif`)** — mide cuánta información sobre la clase aporta
conocer el valor de la variable. Su ventaja sobre las dos anteriores es que **capta relaciones no
lineales**: una variable cuyas dos clases tienen la misma media pero distinta dispersión da d de
Cohen cercano a cero y, sin embargo, información mutua alta. Es la que puede rescatar una variable
que las otras dos descartarían.

Y un cuarto criterio, que no mide relevancia sino **redundancia**: la correlación entre las
predictoras, que ya vimos en el punto 3.8. De dos variables que dicen lo mismo, una sobra.

In [ ]:
# --- Herramientas de selección ---
from sklearn.feature_selection import f_classif, mutual_info_classif

# 1. d de Cohen: ya está calculada en el punto 3.6
ranking = separacion[["variable", "d_cohen", "magnitud", "efecto"]].copy()
ranking = ranking.rename(columns={"magnitud": "d_abs"})

# 2. ANOVA F
f_valores, p_valores = f_classif(df[predictoras], y)
tabla_f = pd.DataFrame({"variable": predictoras, "F": f_valores, "p_valor": p_valores})

# 3. Información mutua (tiene un componente aleatorio: fijamos la semilla)
info_mutua = mutual_info_classif(df[predictoras], y, random_state=SEMILLA)
tabla_im = pd.DataFrame({"variable": predictoras, "info_mutua": info_mutua})

ranking = ranking.merge(tabla_f, on="variable").merge(tabla_im, on="variable")

# Un puesto por criterio, y el promedio de los tres
ranking["puesto_d"] = ranking["d_abs"].rank(ascending=False).astype(int)
ranking["puesto_F"] = ranking["F"].rank(ascending=False).astype(int)
ranking["puesto_IM"] = ranking["info_mutua"].rank(ascending=False).astype(int)
ranking["puesto_medio"] = ranking[["puesto_d", "puesto_F", "puesto_IM"]].mean(axis=1)
ranking = ranking.sort_values("puesto_medio").reset_index(drop=True)
ranking["significativa"] = np.where(ranking["p_valor"] < 0.05, "sí", "no")

print("=" * 96)
print("  RELEVANCIA DE CADA VARIABLE SEGÚN TRES CRITERIOS")
print("=" * 96)
display(ranking[["variable", "d_cohen", "d_abs", "F", "p_valor", "significativa",
                 "info_mutua", "puesto_d", "puesto_F", "puesto_IM", "puesto_medio"]])

print(f"  Variables con p-valor < 0,05 (diferencia estadísticamente significativa): "
      f"{int((ranking['p_valor'] < 0.05).sum())} de {len(predictoras)}")
print()
print("  ¿Coinciden los tres criterios?")
for _, fila in ranking.iterrows():
    puestos = [fila["puesto_d"], fila["puesto_F"], fila["puesto_IM"]]
    desacuerdo = max(puestos) - min(puestos)
    marca = "  <-- los criterios NO coinciden" if desacuerdo >= 2 else ""
    print(f"    {fila['variable']:<12} puestos d={fila['puesto_d']} "
          f"F={fila['puesto_F']} IM={fila['puesto_IM']}{marca}")

Los tres criterios, en un gráfico. Cada panel está normalizado a su propio máximo para que se
puedan comparar las formas, no los valores absolutos —que están en unidades distintas—.

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 4.2))

for eje, (columna, titulo, color) in zip(ejes, [
        ("d_abs", "d de Cohen (valor absoluto)", "#4c72b0"),
        ("F", "ANOVA F", "#55a868"),
        ("info_mutua", "Información mutua", "#c44e52")]):
    datos = ranking.sort_values(columna, ascending=True)
    eje.barh(datos["variable"], datos[columna], color=color,
             edgecolor="black", linewidth=0.6)
    eje.set_title(titulo)
    eje.set_xlabel("")

fig.suptitle("Relevancia de cada variable según tres criterios distintos",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Redundancia: pares de predictoras muy correlacionadas entre sí
print("=" * 72)
print("  REDUNDANCIA ENTRE PREDICTORAS")
print("=" * 72)
if pares_altos:
    for a, b, r in sorted(pares_altos, key=lambda t: -abs(t[2])):
        print(f"    {a:<12} <-> {b:<12}  r = {r:+.3f}   (candidatos a quedarse con una sola)")
else:
    print(f"    Ningún par supera |r| = {UMBRAL_CORR_ALTA}.")
    print("    No hay dos variables que digan lo mismo, así que por redundancia")
    print("    no hay nada que eliminar.")

<a id="sec-6-2"></a>
## 6.2 La decisión, y por qué se toma acá y no antes

**Decisión: se conservan las siete variables.**

Los argumentos, en orden:

1. **Todas aportan algo.** Mirá la tabla del punto 6.1: incluso las peor rankeadas tienen un
   efecto medible. No hay ninguna con d de Cohen cercano a cero *y* F bajo *y* información mutua
   nula, que sería el caso claro de una columna muda.

2. **No hay redundancia.** Ningún par de predictoras supera el umbral de correlación, así que no
   hay dos variables que estén diciendo lo mismo.

3. **Los tres criterios miden cada variable por separado.** Un modelo combina las siete a la vez y
   puede aprovechar información que ninguna tiene sola: dos variables que individualmente separan
   poco pueden separar muy bien en conjunto. Descartar una por su puntaje individual es una
   apuesta, no una conclusión.

4. **Son siete columnas para 171 filas.** La razón es cómoda. La selección de variables se vuelve
   necesaria cuando hay decenas o cientos de columnas, no siete. Acá el costo de conservarlas es
   nulo y el riesgo de descartar señal es real.

**La objeción obvia, y por qué no la aceptamos.**

Mirando la tabla del punto 6.1, alguien podría decir: *hay variables cuyo p-valor supera 0,05, o
sea que su diferencia entre clases no es estadísticamente significativa. ¿Por qué no descartarlas?*

Tres respuestas:

- **El p-valor de ANOVA mide una diferencia de medias, y nada más.** Dos clases pueden tener la
  misma media y distribuciones completamente distintas —una concentrada y la otra dispersa— y
  entonces el test no encuentra nada aunque la variable separe. La columna de información mutua
  está justamente para detectar ese caso, y en la tabla se ve que el orden que da no siempre
  coincide con el de las otras dos.
- **"No significativo" no es lo mismo que "no informativo".** Con 171 filas, el test tiene poca
  potencia: un efecto real pero moderado puede no alcanzar el umbral solo por falta de datos.
  Ausencia de evidencia no es evidencia de ausencia.
- **El p-valor evalúa la variable sola.** El modelo la va a usar junto con las otras seis.

Si igual quisieras descartarlas, el lugar correcto para hacerlo no es acá sino comparando los dos
modelos —con las siete variables y con las cinco— en la Parte 2, con validación cruzada. Ahí la
pregunta deja de ser "¿es significativa?" y pasa a ser "¿el modelo rinde mejor o peor sin ella?",
que es la que realmente importa.

**Y ahora la parte importante, que es un aviso.**

La tabla del punto 6.1 se calculó **mirando las 171 filas, etiquetas incluidas**. Eso está bien
para *entender* los datos, que es lo que estamos haciendo. Pero si la usáramos para **seleccionar**
variables y después evaluáramos el modelo sobre esos mismos datos, estaríamos cometiendo fuga de
información: habríamos elegido las variables que mejor coinciden con las respuestas que después
vamos a tratar de predecir.

No es una sutileza teórica. Con muchas columnas y pocas filas, ese error puede producir modelos
que parecen acertar el 70% sobre datos que son **ruido aleatorio puro**, sin ninguna señal real.

Como acá no descartamos ninguna variable, el problema no llega a materializarse: no hay ninguna
decisión tomada a partir de esa tabla. Pero si el día de mañana quisieras quedarte con las mejores
cinco, la forma correcta no es filtrar el DataFrame acá, sino meter un `SelectKBest` adentro del
pipeline, para que la selección se rehaga con los datos de entrenamiento de cada partición:

```python
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("seleccion", SelectKBest(f_classif, k=5)),
    ("escalador", StandardScaler()),
    ("modelo",    LogisticRegression()),
])
```

Es la misma razón por la que el escalado tampoco se aplica todavía.

In [ ]:
# --- La lista definitiva de variables que van al modelo ---
predictoras_finales = list(predictoras)

print("=" * 72)
print("  VARIABLES SELECCIONADAS")
print("=" * 72)
for i, c in enumerate(predictoras_finales, 1):
    fila = ranking[ranking["variable"] == c].iloc[0]
    print(f"  {i}. {c:<12} d={fila['d_cohen']:+.3f}  F={fila['F']:7.2f}  "
          f"IM={fila['info_mutua']:.4f}  (puesto medio {fila['puesto_medio']:.1f})")
print()
print(f"  Conservadas : {len(predictoras_finales)} de {len(predictoras)}")
print(f"  Descartadas : 0")
print()

X = df[predictoras_finales].copy()

print("  Matrices listas para dividir:")
print(f"    X : {X.shape[0]} filas x {X.shape[1]} columnas  (sin escalar todavía)")
print(f"    y : {y.shape[0]} etiquetas  ->  "
      f"{int((y == 1).sum())} de clase 1 ({CLASE_POSITIVA}), "
      f"{int((y == 0).sum())} de clase 0 ({clase_negativa})")
print()
print(f"    Faltantes en X          : {int(X.isna().sum().sum())}")
print(f"    Faltantes en y          : {int(y.isna().sum())}")
print(f"    Columnas no numéricas   : {list(X.select_dtypes(exclude=[np.number]).columns)}")
print(f"    Filas alineadas X <-> y : {len(X) == len(y)}")

---
# 7. División en entrenamiento y prueba

<a id="sec-7-1"></a>
## 7.1 Por qué hay que dividir

Entrenar un modelo es enseñarle a reconocer patrones. Para saber si **realmente** aprendió, hay
que evaluarlo con datos que no vio durante el entrenamiento.

Si usáramos los mismos datos para las dos cosas, el modelo podría parecer excelente simplemente
porque ya conocía las respuestas. Es como tomarle a alguien un examen con exactamente las mismas
preguntas que estudió: un 10 no prueba que haya entendido el tema, solo que tiene buena memoria.

Por eso se parte el dataset en dos:

- **Conjunto de entrenamiento** (*training set*) — con el que el modelo ajusta sus parámetros
  internos. Suele ser el 70–80%.
- **Conjunto de prueba** (*test set*) — reservado, se usa **una sola vez**, al final, para estimar
  cómo se comportaría con datos nuevos. Simula el mundo real.

Las proporciones habituales son 80/20, 70/30 o 90/10, según el tamaño del dataset. Acá usamos
80/20.

**Dos observaciones propias, para no vender la división como más sólida de lo que es.**

La primera: con 171 filas, el 20% son **34 casos**. Cada acierto o error mueve el accuracy casi 3
puntos porcentuales, y el resultado depende de qué 34 casos hayan caído del lado de la prueba, que
es cosa del azar. Una sola división da un número, no una estimación confiable. En la Parte 2 esto
se resuelve con **validación cruzada**, que usa las 171 filas para evaluar en lugar de 34.

La segunda: esta división es el **punto de no retorno** del cuaderno. A partir de acá, todo lo que
se calcule para transformar los datos tiene que calcularse con el conjunto de entrenamiento
solamente. Es lo que hacemos en el punto 7.3.

<a id="sec-7-2"></a>
## 7.2 La división 80/20 estratificada

Dos detalles de implementación que importan:

- **`stratify=y`** mantiene la misma proporción de clases en las dos partes. Sin esto, el azar
  podría dejar el conjunto de prueba con muchos más casos de una clase que de la otra, y la
  evaluación quedaría distorsionada. Con clases desbalanceadas es prácticamente obligatorio.
- **`random_state=SEMILLA`** fija la aleatoriedad para que la división sea siempre la misma y los
  resultados se puedan reproducir. Sin esto, el cuaderno da números distintos en cada corrida y no
  se puede comparar nada.

In [ ]:
# --- Herramienta de división ---
from sklearn.model_selection import train_test_split

X_entrena, X_prueba, y_entrena, y_prueba = train_test_split(
    X, y,
    test_size=PROPORCION_PRUEBA,
    stratify=y,
    random_state=SEMILLA,
)

print("=" * 78)
print(f"  DIVISIÓN {1 - PROPORCION_PRUEBA:.0%} / {PROPORCION_PRUEBA:.0%} ESTRATIFICADA")
print("=" * 78)
print(f"  Entrenamiento : {len(X_entrena):>4} casos  ({len(X_entrena)/len(X):.1%})")
print(f"  Prueba        : {len(X_prueba):>4} casos  ({len(X_prueba)/len(X):.1%})")
print(f"  Total         : {len(X_entrena) + len(X_prueba):>4} casos")
print()
print("  Reparto de clases — la estratificación lo mantiene parejo:")
print(f"    {'':>18} {CLASE_POSITIVA:>8} {clase_negativa:>8} {'% positiva':>13}")
for nombre, yy in [("Dataset completo", y), ("Entrenamiento", y_entrena), ("Prueba", y_prueba)]:
    print(f"    {nombre:>18} {int((yy == 1).sum()):>8} {int((yy == 0).sum()):>8} "
          f"{(yy == 1).mean():>12.1%}")

print()
print("  VERIFICACIONES")
print("  " + "-" * 74)
sin_perdidas = len(X_entrena) + len(X_prueba) == len(X)
sin_solape = len(set(X_entrena.index) & set(X_prueba.index)) == 0
desvio_estrato = abs(y_entrena.mean() - y_prueba.mean())
print(f"    No se perdió ninguna fila           : {sin_perdidas}")
print(f"    No hay filas en los dos conjuntos   : {sin_solape}")
print(f"    Diferencia de % positiva entre partes: {desvio_estrato:.3f}")
print(f"    Las dos clases están en los dos     : "
      f"{y_entrena.nunique() == 2 and y_prueba.nunique() == 2}")

fig, ejes = plt.subplots(1, 2, figsize=(13, 4))

ejes[0].bar(["Entrenamiento", "Prueba"], [len(X_entrena), len(X_prueba)],
            color=["#4c72b0", "#dd8452"], edgecolor="black", linewidth=0.7)
for i, v in enumerate([len(X_entrena), len(X_prueba)]):
    ejes[0].text(i, v + 2, f"{v} casos", ha="center", fontweight="bold")
ejes[0].set_title("Tamaño de cada conjunto")
ejes[0].set_ylabel("casos")
ejes[0].set_ylim(0, max(len(X_entrena), len(X_prueba)) * 1.18)

anchura = 0.35
posiciones = np.arange(2)
positivos = [int((y_entrena == 1).sum()), int((y_prueba == 1).sum())]
negativos = [int((y_entrena == 0).sum()), int((y_prueba == 0).sum())]
ejes[1].bar(posiciones - anchura/2, positivos, anchura, label=CLASE_POSITIVA,
            color=COLOR_CLASE[CLASE_POSITIVA], edgecolor="black", linewidth=0.6)
ejes[1].bar(posiciones + anchura/2, negativos, anchura, label=clase_negativa,
            color=COLOR_CLASE[clase_negativa], edgecolor="black", linewidth=0.6)
ejes[1].set_xticks(posiciones)
ejes[1].set_xticklabels(["Entrenamiento", "Prueba"])
ejes[1].set_title("Reparto de clases en cada conjunto")
ejes[1].set_ylabel("casos")
ejes[1].legend(title=objetivo)

plt.tight_layout()
plt.show()

<a id="sec-7-3"></a>
## 7.3 Recién ahora: aplicar el escalado sin fuga de información

Acá se cierra lo que quedó pendiente en el punto 5.2.

Un escalador **aprende parámetros de los datos**: Min-Max aprende el mínimo y el máximo, Z-score
aprende la media y el desvío. La regla es corta y no tiene excepciones:

| Método | Sobre qué conjunto | Qué hace |
|---|---|---|
| `fit_transform()` | **Solo** entrenamiento | Aprende los parámetros y transforma |
| `transform()` | Prueba | Aplica los parámetros ya aprendidos, sin volver a aprender |

Si en lugar de eso se hiciera `fit_transform()` sobre el dataset completo antes de dividir, los
parámetros contendrían los valores del conjunto de prueba. El modelo recibiría, de forma
indirecta, información sobre los datos que después lo van a evaluar, y la evaluación saldría
**optimista**: mejor de lo que el modelo realmente es. Lo peor del caso es que no hay ningún
error, ningún aviso y nada que mirar. Solo un número más lindo de lo que corresponde.

La celda de abajo hace las dos cosas —la correcta y la incorrecta— y compara los parámetros, para
que se vea de dónde sale la diferencia.

**Nota sobre el escalador elegido.** Aplicamos los dos, Min-Max y Z-score, y dejamos las cuatro
matrices listas. Cuál conviene depende del modelo, y eso se decide en la Parte 2; dejar los dos
preparados no cuesta nada y evita tener que volver acá.

**Una advertencia para la Parte 2.** Hacer esto a mano, como acá, funciona para una división
única. En cuanto aparezca la validación cruzada, la forma robusta es meter el escalador adentro de
un `Pipeline` de scikit-learn, que se encarga de reajustarlo en cada partición automáticamente. La
fuga deja de ser un error posible y pasa a ser imposible.

In [ ]:
# --- LA FORMA CORRECTA: el escalador solo ve el entrenamiento ---
escalador_z = StandardScaler()
X_entrena_z = pd.DataFrame(escalador_z.fit_transform(X_entrena),
                           columns=predictoras_finales, index=X_entrena.index)
X_prueba_z = pd.DataFrame(escalador_z.transform(X_prueba),
                          columns=predictoras_finales, index=X_prueba.index)

escalador_mm = MinMaxScaler()
X_entrena_mm = pd.DataFrame(escalador_mm.fit_transform(X_entrena),
                            columns=predictoras_finales, index=X_entrena.index)
X_prueba_mm = pd.DataFrame(escalador_mm.transform(X_prueba),
                           columns=predictoras_finales, index=X_prueba.index)

print("=" * 78)
print("  ESCALADO APLICADO — fit solo con entrenamiento, transform en prueba")
print("=" * 78)
print(f"  X_entrena_z  : {X_entrena_z.shape}   media {X_entrena_z.to_numpy().mean():+.6f}  "
      f"desvío {X_entrena_z.to_numpy().std():.4f}")
print(f"  X_prueba_z   : {X_prueba_z.shape}    media {X_prueba_z.to_numpy().mean():+.6f}  "
      f"desvío {X_prueba_z.to_numpy().std():.4f}")
print()
print("  Notar que el conjunto de PRUEBA no queda con media exactamente 0 ni desvío 1.")
print("  Eso es correcto y esperable: se transformó con los parámetros del")
print("  entrenamiento, no con los propios. Si diera exactamente 0 y 1, sería la")
print("  señal de que se le hizo fit por error.")
print()
print(f"  X_entrena_mm : rango [{X_entrena_mm.to_numpy().min():.4f}, "
      f"{X_entrena_mm.to_numpy().max():.4f}]")
print(f"  X_prueba_mm  : rango [{X_prueba_mm.to_numpy().min():.4f}, "
      f"{X_prueba_mm.to_numpy().max():.4f}]")
print()
print("  Tampoco el de prueba queda exactamente entre 0 y 1: un valor más alto que")
print("  el máximo del entrenamiento se escala por encima de 1, y uno más bajo que")
print("  el mínimo, por debajo de 0. Es correcto, y es exactamente lo que va a pasar")
print("  en producción cada vez que llegue un caso fuera del rango ya visto.")

# --- LA FORMA INCORRECTA, solo para ver la diferencia ---
escalador_mal = StandardScaler().fit(X)      # <-- ve TODO, incluido el test

comparacion_fuga = pd.DataFrame({
    "media_con_todo_MAL": escalador_mal.mean_,
    "media_solo_entrena_BIEN": escalador_z.mean_,
    "dif_medias": escalador_mal.mean_ - escalador_z.mean_,
    "desvio_con_todo_MAL": np.sqrt(escalador_mal.var_),
    "desvio_solo_entrena_BIEN": np.sqrt(escalador_z.var_),
}, index=predictoras_finales)
comparacion_fuga["dif_desvios"] = (comparacion_fuga["desvio_con_todo_MAL"] -
                                   comparacion_fuga["desvio_solo_entrena_BIEN"])

print()
print("=" * 88)
print("  DE DÓNDE SALE LA FUGA: los parámetros cambian según qué datos vio el escalador")
print("=" * 88)
display(comparacion_fuga)
print("  Esas diferencias son información del conjunto de prueba filtrándose hacia")
print("  la transformación que se aplica al entrenamiento. Parecen chicas, pero el")
print("  efecto crece cuanto más chico es el dataset y más extremos tiene.")

---
# 8. Exportar los resultados

<a id="sec-8-1"></a>
## 8.1 Los archivos que deja el cuaderno

Así funciona un pipeline de datos real: cada etapa consume lo que produjo la anterior y deja su
propio artefacto. Tiene tres ventajas concretas:

1. **La etapa siguiente no repite el trabajo.** La Parte 2 no tiene que volver a auditar lo que ya
   se auditó acá.
2. **Queda un registro de lo que se decidió.** Los archivos son el estado de los datos después de
   cada transformación, con fecha y hora.
3. **Si mañana cambia la limpieza**, se vuelve a correr este cuaderno y se regeneran. La Parte 2
   queda igual.

Exportamos en **CSV** y no en Excel por tres razones: lo lee cualquier herramienta sin librerías
extra, pesa menos, y no guarda formatos ni fórmulas que puedan confundirse con datos. El único
cuidado es usar `index=False`, para que Pandas no agregue una columna con el número de fila que
después aparecería como una variable más llamada `Unnamed: 0`.

| Archivo | Qué contiene |
|---|---|
| `dataset_limpio.csv` | Las 171 filas auditadas, sin transformar |
| `entrenamiento.csv` | El 80%, con la etiqueta codificada |
| `prueba.csv` | El 20%, con la etiqueta codificada |
| `metadatos_limpieza.txt` | El registro de qué se revisó y qué se decidió |

Los conjuntos se exportan **sin escalar**, a propósito: el escalado depende del modelo que se vaya
a usar, y los parámetros del escalador quedan en el cuaderno. Guardar los datos crudos deja todas
las opciones abiertas.

In [ ]:
from datetime import datetime

# --- 1. El dataset auditado, tal como salió del punto 4 ---
df_limpio = df.copy()
df_limpio.to_csv(ARCHIVO_LIMPIO, index=False, encoding="utf-8")

# --- 2. Los conjuntos de entrenamiento y prueba, con la etiqueta codificada ---
entrenamiento = X_entrena.copy()
entrenamiento[objetivo] = df.loc[X_entrena.index, objetivo]
entrenamiento["y"] = y_entrena
entrenamiento.to_csv(ARCHIVO_ENTRENA, index=False, encoding="utf-8")

prueba = X_prueba.copy()
prueba[objetivo] = df.loc[X_prueba.index, objetivo]
prueba["y"] = y_prueba
prueba.to_csv(ARCHIVO_PRUEBA, index=False, encoding="utf-8")

# --- 3. Los metadatos del proceso ---
ARCHIVO_METADATOS = "metadatos_limpieza.txt"
with open(ARCHIVO_METADATOS, "w", encoding="utf-8") as archivo:
    archivo.write("METADATOS DEL PREPROCESAMIENTO — Parte 1\n")
    archivo.write("=" * 66 + "\n")
    archivo.write(f"Fecha                 : {datetime.now():%Y-%m-%d %H:%M:%S}\n")
    archivo.write(f"Archivo de origen     : {RUTA_ARCHIVO}\n")
    archivo.write(f"Filas                 : {len(df_limpio)}\n")
    archivo.write(f"Columnas              : {df_limpio.shape[1]}\n")
    archivo.write(f"Variable objetivo     : {objetivo}\n")
    archivo.write(f"Clases                : {sorted(df_limpio[objetivo].unique())}\n")
    archivo.write(f"Codificación          : {mapa_clases}\n")
    archivo.write(f"Variables predictoras : {predictoras_finales}\n")
    archivo.write(f"Variables descartadas : ninguna\n")
    archivo.write(f"Semilla               : {SEMILLA}\n")
    archivo.write(f"División              : {1-PROPORCION_PRUEBA:.0%} entrenamiento / "
                  f"{PROPORCION_PRUEBA:.0%} prueba, estratificada\n")
    archivo.write(f"Entrenamiento         : {len(X_entrena)} casos\n")
    archivo.write(f"Prueba                : {len(X_prueba)} casos\n")
    archivo.write("\nAuditoría de calidad (punto 4):\n")
    for _, fila in auditoria.iterrows():
        archivo.write(f"  {fila['control']:<52} {fila['encontrado']:<22} {fila['accion']}\n")
    archivo.write("\nDecisiones tomadas:\n")
    archivo.write("  - No se eliminó ni se modificó ningún dato.\n")
    archivo.write("  - Los outliers se detectaron y se conservaron (justificado en 4.5).\n")
    archivo.write("  - No se agregaron variables derivadas (justificado en 5.3).\n")
    archivo.write("  - No se descartó ninguna variable (justificado en 6.2).\n")
    archivo.write("  - El escalado se ajustó SOLO con el conjunto de entrenamiento (7.3).\n")

SALIDAS = [ARCHIVO_LIMPIO, ARCHIVO_ENTRENA, ARCHIVO_PRUEBA, ARCHIVO_METADATOS]

print("=" * 78)
print("  ARCHIVOS GENERADOS")
print("=" * 78)
for nombre, objeto in [(ARCHIVO_LIMPIO, df_limpio),
                       (ARCHIVO_ENTRENA, entrenamiento),
                       (ARCHIVO_PRUEBA, prueba)]:
    print(f"  {nombre:<26} {objeto.shape[0]:>4} filas x {objeto.shape[1]} columnas")
print(f"  {ARCHIVO_METADATOS:<26} registro del proceso")
print()

# --- Verificación: releemos lo que escribimos ---
control = pd.read_csv(ARCHIVO_LIMPIO)
control_tr = pd.read_csv(ARCHIVO_ENTRENA)
control_te = pd.read_csv(ARCHIVO_PRUEBA)

print("  VERIFICACIÓN DE LA ESCRITURA (releemos los archivos)")
print("  " + "-" * 74)
print(f"    {ARCHIVO_LIMPIO:<22} dimensiones iguales : {control.shape == df_limpio.shape}")
print(f"    {ARCHIVO_ENTRENA:<22} dimensiones iguales : {control_tr.shape == entrenamiento.shape}")
print(f"    {ARCHIVO_PRUEBA:<22} dimensiones iguales : {control_te.shape == prueba.shape}")
print(f"    Suma de filas = original                  : "
      f"{len(control_tr) + len(control_te) == len(control)}")
print(f"    Sin nulos en ninguno                      : "
      f"{int(control.isna().sum().sum() + control_tr.isna().sum().sum() + control_te.isna().sum().sum()) == 0}")
print()
print("  Primeras filas del conjunto de entrenamiento:")
display(control_tr.head(3))

<a id="sec-8-2"></a>
## 8.2 Cómo llevárselos

Si estás en Colab, los archivos se escribieron en el disco de la máquina virtual, que es
**temporal**: cuando la sesión se desconecta, desaparecen. Hay que moverlos a algún lado que
persista.

| Si la Parte 2 la vas a correr... | Guardá los archivos en... |
|---|---|
| En Colab, más tarde o en otra sesión | Google Drive |
| En tu computadora, con Jupyter o VS Code | Descargados a tu disco |
| Ahora mismo, sin cerrar la sesión | Donde están; la máquina virtual los conserva |

Descomentá la opción que corresponda.

In [ ]:
# ---------------------------------------------------------------------
# LLEVARSE LOS ARCHIVOS
# ---------------------------------------------------------------------
# La lista SALIDAS quedó armada en la celda anterior.

# --- Opción 1: copiarlos a Google Drive ---
# import shutil, os
# from google.colab import drive
# drive.mount("/content/drive")
#
# CARPETA_DESTINO = "/content/drive/MyDrive/datasets"
# os.makedirs(CARPETA_DESTINO, exist_ok=True)
# for archivo in SALIDAS:
#     shutil.copy(archivo, CARPETA_DESTINO)
#     print(f"Copiado: {archivo}")

# --- Opción 2: descargarlos a tu computadora ---
# from google.colab import files
# for archivo in SALIDAS:
#     files.download(archivo)

# --- Opción 3: empaquetar todo en un zip y bajar uno solo ---
# import shutil
# shutil.make_archive("salidas_parte1", "zip", ".", base_dir=None)
# from google.colab import files
# files.download("salidas_parte1.zip")

print("Archivos listos:")
for archivo in SALIDAS:
    print(f"    {archivo}")
print()
print("Si estás en Colab, acordate de llevártelos antes de cerrar la sesión.")

---
<a id="sec-cierre"></a>
# Cierre — Resumen y observaciones finales

**El recorrido**

| Punto | Qué se hizo |
|---|---|
| 0.2 | Tres formas de cargar el archivo en Colab, con sus ventajas y sus trampas |
| 1 | Selección del dataset: origen, problema y justificación |
| 2 | Estructura, tipos y estadísticos descriptivos |
| 3 | EDA: objetivo, distribuciones, densidad por clase, correlaciones, un interactivo |
| 4 | Auditoría de calidad en cinco controles |
| 5 | Transformaciones: codificación de la etiqueta, escaladores, variables derivadas |
| 6 | Selección de variables con tres criterios |
| 7 | División 80/20 estratificada y escalado sin fuga de información |
| 8 | Exportación de los cuatro archivos |

**Las decisiones que se tomaron, y por qué**

1. **No se eliminó ningún dato.** Ni outliers ni casos raros: la auditoría mostró que todos los
   valores son válidos, y con 171 filas cada caso pesa demasiado como para descartarlo sin una
   razón fuerte. Justificado en 4.5.
2. **No se agregaron variables derivadas.** Se construyeron, se midieron y se descartaron: no
   traían información que no estuviera ya en las siete originales. Justificado en 5.3.
3. **No se descartó ninguna variable.** Las tres medidas de relevancia coinciden en que todas
   aportan algo, y no hay redundancia entre ellas. Justificado en 6.2.
4. **El escalado se aplicó después de dividir**, ajustándolo solo con el conjunto de
   entrenamiento. Justificado en 5.2 y aplicado en 7.3.

**Observaciones propias sobre lo que se vio**

`[Completar con lo que más te haya llamado la atención. Algunas líneas para arrancar:]`

- `[El dataset llegó sorprendentemente limpio: cero nulos, cero duplicados, cero tipos mal
  leídos. Eso no es lo habitual y conviene decirlo, porque explica por qué la etapa de limpieza
  terminó sin modificar nada.]`
- `[La densidad por clase del punto 3.6 muestra que N de las siete variables separan bien las
  clases y las otras casi no. Eso anticipa que un modelo lineal simple puede andar razonablemente
  bien.]`
- `[Las correlaciones son casi todas negativas, y no por una relación real entre las variables
  sino porque las siete reparten un total fijo de 1. Es el tipo de cosa que se interpreta mal si
  uno no sabe cómo se construyeron los datos.]`
- `[Otra observación propia.]`

**Limitaciones que hay que tener presentes en la Parte 2**

| Limitación | Consecuencia |
|---|---|
| 171 filas | El conjunto de prueba son 34 casos: la estimación es ruidosa. Hace falta validación cruzada |
| Las siete variables suman 1 | No son independientes; los coeficientes de un modelo lineal hay que leerlos como pesos relativos |
| `[Completar]` | |

**Qué sigue**

La Parte 2 lee `dataset_limpio.csv` y entrena cuatro modelos de clasificación —Regresión
Logística, KNN, Árbol de Decisión y Random Forest— evaluados todos con el mismo bloque de
métricas, con validación cruzada, y los compara en un ranking final.

In [ ]:
print("=" * 78)
print("  PARTE 1 COMPLETADA")
print("=" * 78)
print(f"  Archivo de origen       : {RUTA_ARCHIVO}")
print(f"  Filas analizadas        : {len(df)}")
print(f"  Variables predictoras   : {len(predictoras_finales)}")
print(f"  Clases                  : {dict(conteo)}")
print()
print(f"  Datos eliminados        : 0")
print(f"  Datos modificados       : 0")
print(f"  Variables creadas       : 0")
print(f"  Variables descartadas   : 0")
print()
print(f"  Entrenamiento           : {len(X_entrena)} casos")
print(f"  Prueba                  : {len(X_prueba)} casos")
print(f"  Escaladores ajustados   : Z-score y Min-Max, solo con entrenamiento")
print()
print(f"  Archivos para la Parte 2:")
for archivo in SALIDAS:
    print(f"      {archivo}")
print("=" * 78)